# Task 1 — Reasoning Trajectories: удобная форма для эксперта (Google Colab)

Этот блокнот сделан для экспертов, которые **не обязаны знать программирование**.

Вы заполняете интерактивную форму → блокнот сам собирает корректный **YAML‑файл** по шаблону Task 1.

Что поддерживается:
- Шаги **1…N** (без верхнего ограничения)
- В каждом шаге — **любое число источников**: text / image / table
- В конце — **связи между шагами** как ориентированный граф (edges)
- Домен выбирается **только через Wikidata** и сохраняется как QID (например `Q336`)
- Форма автосохраняется во время работы и может восстановить введённые значения после повторного запуска ячейки
- В блоке связей шаги выбираются обычным кликом мышки; повторный клик снимает выделение


# Пошаговый туториал (подробно)

## Шаг 0. Как не «сломать» ничего
- В этом блокноте **не нужно редактировать код**.
- Делайте только 2 действия:
  1) запускать ячейки (▶ / Shift+Enter)
  2) заполнять форму и нажимать кнопки

---

## Шаг 1. Запустите ячейку «Установка и импорт»
Она установит нужные библиотеки (ipywidgets, PyYAML, requests) и подготовит окружение.

---

## Шаг 2. Запустите ячейку «Форма»
Появится интерфейс с блоками:
1) **Эксперт** (ФИО)
2) **Основное** (topic, cutoff_year, domain через Wikidata)
3) **Публикации**
4) **Шаги (1…N)** + источники
5) **Связи** между шагами (ориентированный граф)
6) **Экспорт** (валидация + сохранение YAML)
7) **GitHub (опционально)** — загрузка YAML в репозиторий одним коммитом

---

## Домен (Wikidata → QID)
1) Введите запрос (например: *Science*, *Lithium-ion battery*, *Наука*)
2) Нажмите **«Найти в Wikidata»**
3) Выберите подходящую сущность в списке

В YAML сохранится **только QID** (например `Q336`).

---

## Шаги и источники
- Добавляйте шаги кнопкой **«➕ Добавить шаг»** (можно 1, 2, 50…)
- В каждом шаге можно добавить сколько угодно источников:
  - type = text / image / table
  - source = id статьи (doi/arxiv/openalex/…) или URL
  - snippet* = цитата / краткая выжимка
  - page/locator — заполняйте, если применимо (особенно для figure/table)

---

## Связи (edges) — ориентированный граф
В блоке «Связи» для каждого шага **i →** просто щёлкните по нужным шагам.
Повторный щелчок снимает связь — `Ctrl/Cmd` не нужен.
На выходе будет список ребер вида `[(1, 2), (1, 3)]` (ориентированные пары).

---

## Экспорт и отправка
1) Нажмите **«Проверить и собрать YAML»** — увидите ошибки (если есть) и превью.
2) Если в форме есть ошибки, введённые значения **не сбросятся**: форма автосохраняет текущее состояние.
3) Файл сохранится рядом с блокнотом.
4) Если нужно — заполните блок **GitHub** и нажмите **«Загрузить YAML в GitHub»** (это создаст коммит в выбранной ветке).



In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
!pip -q install ipywidgets pyyaml requests unidecode

import os, re, json, base64, textwrap, datetime, hashlib
from pathlib import Path

import yaml
import requests
import ipywidgets as W
from IPython.display import display, Markdown, clear_output

# Для транскрипции имён (кириллица/другие алфавиты → латиница)
try:
    from unidecode import unidecode
except Exception:
    unidecode = None

# (Опционально) для скачивания/Drive в Colab
try:
    from google.colab import files, drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("Готово. Теперь запускайте следующую ячейку: «Вспомогательные функции».")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.5 MB/s eta 0:00:00
Готово. Теперь запускайте следующую ячейку: «Вспомогательные функции».


In [ ]:
# @title
# Вспомогательные функции (не нужно редактировать)

# --- Wikidata: поиск домена по справочнику, чтобы все эксперты выбирали одно и то же ---
_WIKIDATA_SEARCH_CACHE = {}

# Викимедиа может возвращать 403, если User-Agent отсутствует или слишком общий.
# По политике Wikimedia желательно указывать понятный User-Agent (и по возможности контакт).
WIKIDATA_USER_AGENT = "TrajectoryFormColab/1.0 (mailto:your-email@example.com)"

def wikidata_search(query: str, language: str = "ru", limit: int = 10):
    """Ищет сущности в Wikidata по строке query."""
    query = (query or "").strip()
    language = (language or "ru").strip() or "ru"
    limit = int(limit or 10)

    if not query:
        return []

    cache_key = (query.lower(), language, limit)
    if cache_key in _WIKIDATA_SEARCH_CACHE:
        return _WIKIDATA_SEARCH_CACHE[cache_key]

    params = {
        "action": "wbsearchentities",
        "search": query,
        "language": language,
        "format": "json",
        "limit": limit,
    }

    headers = {
        "User-Agent": WIKIDATA_USER_AGENT,
        "Accept": "application/json",
    }

    urls = ["https://www.wikidata.org/w/api.php", "https://wikidata.org/w/api.php"]

    last_exc = None
    for url in urls:
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 403 and url != urls[-1]:
                continue
            r.raise_for_status()
            data = r.json() or {}

            out = []
            for item in (data.get("search") or [])[:limit]:
                qid = item.get("id")
                out.append({
                    "qid": qid,
                    "label": item.get("label"),
                    "description": item.get("description"),
                    "url": item.get("concepturi") or (f"https://www.wikidata.org/wiki/{qid}" if qid else None),
                })

            _WIKIDATA_SEARCH_CACHE[cache_key] = out
            return out

        except Exception as e:
            last_exc = e

    raise last_exc


# -----------------------------
# Идентификатор эксперта: ФИО → латиница + хэш артефакта
# -----------------------------

_CYR = {
    "а":"a","б":"b","в":"v","г":"g","д":"d","е":"e","ё":"e","ж":"zh","з":"z","и":"i","й":"i",
    "к":"k","л":"l","м":"m","н":"n","о":"o","п":"p","р":"r","с":"s","т":"t","у":"u","ф":"f",
    "х":"kh","ц":"ts","ч":"ch","ш":"sh","щ":"shch","ы":"y","э":"e","ю":"yu","я":"ya","ь":"","ъ":"",
}

def _transliterate_to_latin(text: str) -> str:
    """Транскрибирует строку в латиницу (по возможности)."""
    text = (text or "").strip()
    if not text:
        return ""
    if unidecode is not None:
        try:
            return unidecode(text)
        except Exception:
            pass
    out = []
    for ch in text:
        low = ch.lower()
        if low in _CYR:
            rep = _CYR[low]
            out.append(rep.capitalize() if ch.isupper() and rep else rep)
        else:
            out.append(ch)
    return "".join(out)

def _slugify(text: str, max_len: int = 80) -> str:
    text = (text or "").strip().lower()
    text = _transliterate_to_latin(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return (text[:max_len] or "trajectory").strip("_")

def _expert_name_parts(last_name: str, first_name: str, patronymic: str):
    last_name = (last_name or "").strip()
    first_name = (first_name or "").strip()
    patronymic = (patronymic or "").strip()
    full = " ".join([p for p in [last_name, first_name, patronymic] if p])
    latin = " ".join([p for p in [
        _transliterate_to_latin(last_name).strip(),
        _transliterate_to_latin(first_name).strip(),
        _transliterate_to_latin(patronymic).strip(),
    ] if p])
    return full, latin

def _expert_prefix_slug(last_name: str, first_name: str, patronymic: str) -> str:
    full, latin = _expert_name_parts(last_name, first_name, patronymic)
    base = latin or full
    return _slugify(base, max_len=60)

def _artifact_hash_for_doc(doc: dict) -> str:
    tmp = dict(doc or {})
    tmp.pop("artifact_hash", None)
    tmp.pop("submission_id", None)
    text = yaml.safe_dump(tmp, sort_keys=False, allow_unicode=True)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def finalize_doc_with_ids(doc: dict, last_name: str, first_name: str, patronymic: str) -> dict:
    doc = dict(doc or {})
    full, latin = _expert_name_parts(last_name, first_name, patronymic)

    expert = {
        "last_name": (last_name or "").strip(),
        "first_name": (first_name or "").strip(),
        "patronymic": (patronymic or "").strip(),
        "full_name": full,
        "latin_full_name": latin,
        "latin_slug": _expert_prefix_slug(last_name, first_name, patronymic),
    }
    doc["expert"] = expert

    h = _artifact_hash_for_doc(doc)
    prefix = expert["latin_slug"] or "expert"
    doc["artifact_hash"] = h
    doc["submission_id"] = f"{prefix}__{h}"
    return doc


# -----------------------------
# Поиск доменов (из configs/domains)
# -----------------------------

def _find_repo_root(start=None):
    start = Path(start or Path.cwd())
    for p in [start, *start.parents]:
        if (p / "configs" / "domains").exists():
            return p
    return None

def _load_domain_configs():
    # Ищем домены в:
    #   - ./configs/domains/*.yaml
    #   - ./examples/**/configs/domains/*.yaml
    # Если репозиторий не найден — возвращаем базовый список.
    repo = _find_repo_root()
    configs = []
    paths = []
    if repo:
        paths += list((repo / "configs" / "domains").glob("*.yaml"))
        paths += list((repo / "examples").glob("**/configs/domains/*.yaml"))
    for path in paths:
        try:
            doc = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
            domain_id = str(doc.get("domain_id") or "").strip()
            title = str(doc.get("title") or domain_id).strip()
            wikidata_qid = str(doc.get("wikidata_qid") or "").strip()
            req = (doc.get("artifact_validation") or {}).get("trajectory_required_conditions") or []
            req = [str(x) for x in req] if isinstance(req, list) else []
            if domain_id:
                configs.append({
                    "domain_id": domain_id,
                    "title": title,
                    "wikidata_qid": wikidata_qid,
                    "required_conditions": req,
                    "source_path": str(path),
                })
        except Exception:
            pass

    if not configs:
        configs = [
            {"domain_id": "science", "title": "General scientific literature", "wikidata_qid": "Q336", "required_conditions": [], "source_path": "embedded"},
        ]

    configs = sorted(configs, key=lambda x: (x["domain_id"] != "science", x["domain_id"]))
    return configs


def _now_utc_z():
    dt = datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0)
    return dt.isoformat().replace("+00:00", "Z")


# -----------------------------
# Google Form: отправка YAML (best effort)
# -----------------------------

GOOGLE_FORM_DEFAULT_URL = "https://forms.gle/aWszGzcKcnai2UXv9"

def google_form_response_url(view_or_short_url: str) -> str:
    url = (view_or_short_url or "").strip()
    if not url:
        raise ValueError("Пустой URL формы")
    try:
        r = requests.get(url, allow_redirects=True, timeout=30, headers={"User-Agent": WIKIDATA_USER_AGENT})
        url = r.url or url
    except Exception:
        pass
    if "/viewform" in url:
        return url.split("/viewform", 1)[0] + "/formResponse"
    return url

def submit_to_google_form(form_url: str, entry_map: dict, values: dict):
    if not isinstance(entry_map, dict) or not entry_map:
        raise ValueError(
            "Не задана карта полей entry.* для формы.\n"
            "⚠️ Авто-отправка в Google Form работает только для ТЕКСТОВЫХ вопросов (не File Upload).\n"
            "Если вы владелец формы: получите «предварительно заполненную ссылку» и возьмите из неё entry.XXXX для нужных вопросов.\n"
            "Иначе используйте ручную отправку: откройте форму и прикрепите YAML файл."
        )

    resp_url = google_form_response_url(form_url)
    if "/formResponse" not in resp_url:
        raise ValueError("Не удалось получить formResponse URL. Попробуйте открыть форму вручную.")

    payload = {}
    for k, entry in entry_map.items():
        if not entry:
            continue
        payload[str(entry)] = str(values.get(k, ""))

    headers = {"User-Agent": WIKIDATA_USER_AGENT, "Referer": form_url}
    r = requests.post(resp_url, data=payload, headers=headers, timeout=60)
    if r.status_code >= 400:
        raise ValueError(
            f"Google Form вернул статус {r.status_code}. "
            "Возможные причины: форма требует авторизацию / в форме есть File Upload вопрос / сеть блокирует запрос."
        )
    return r.status_code


# -----------------------------
# Валидация (понятные ошибки)
# -----------------------------

def _validate_trajectory(doc: dict, domain_required):
    """Validate trajectory artifact.

    New schema (artifact_version>=2):
      - domain: Wikidata QID (e.g. Q336)
      - steps[].sources: list of sources (text/image/table)
      - edges: optional list of pairs (from_step_id, to_step_id)

    Backward compatibility:
      - steps[].evidence (single dict) is accepted and treated as a one-element sources list.
    """

    def err(path, message, hint=None, *, kind="error", step_index=None, widget_key=None):
        return {
            "path": path,
            "message": message,
            "hint": hint,
            "kind": kind,
            "step_index": step_index,
            "widget_key": widget_key,
        }

    def _normalize_sources(step: dict):
        if isinstance(step.get("sources"), list):
            return step.get("sources") or []
        ev = step.get("evidence")
        if isinstance(ev, dict) and ev:
            t = str(ev.get("type") or "text").strip().lower()
            if t == "figure":
                t = "image"
            return [{
                "type": t,
                "source": ev.get("source", ""),
                "page": ev.get("page", ""),
                "locator": ev.get("figure_or_table", "") or "",
                "snippet_or_summary": ev.get("snippet_or_summary", ""),
            }]
        return []

    errs = []

    expert = doc.get("expert") or {}
    if not isinstance(expert, dict):
        expert = {}
    if not str(expert.get("last_name") or "").strip():
        errs.append(err("expert.last_name", "Обязательное поле: фамилия.", "Заполните фамилию.", widget_key="expert.last_name"))
    if not str(expert.get("first_name") or "").strip():
        errs.append(err("expert.first_name", "Обязательное поле: имя.", "Заполните имя.", widget_key="expert.first_name"))
    if not str(expert.get("patronymic") or "").strip():
        errs.append(err("expert.patronymic", "Обязательное поле: отчество.", "Если отчества нет — поставьте '-'.", widget_key="expert.patronymic"))

    steps = doc.get("steps", [])
    if not isinstance(steps, list) or len(steps) < 1:
        errs.append(err("steps", "Нужно минимум 1 шаг.", "Добавьте хотя бы один шаг."))

    papers = doc.get("papers", [])
    if not isinstance(papers, list) or len(papers) == 0:
        errs.append(err("papers", "Нужно добавить минимум 1 публикацию.", "Добавьте хотя бы одну публикацию (id/year/title)."))

    if not str(doc.get("topic") or "").strip():
        errs.append(err("topic", "Обязательное поле.", "Заполните Topic (тема траектории).", widget_key="topic"))

    domain_qid = str(doc.get("domain") or "").strip()
    if not domain_qid:
        errs.append(err("domain", "Обязательное поле.", "Выберите домен через Wikidata (QID).", widget_key="domain"))
    elif not re.fullmatch(r"Q\d+", domain_qid):
        errs.append(err("domain", "Домен должен быть Wikidata QID вида Q12345.", "Пример: Q336.", widget_key="domain"))

    if not isinstance(doc.get("cutoff_year", None), int):
        errs.append(err("cutoff_year", "Должен быть числом (год).", "Пример: 2020.", widget_key="cutoff_year"))

    # Validate step_ids + build set
    step_ids = []
    for step in steps if isinstance(steps, list) else []:
        if isinstance(step, dict) and step.get("step_id") is not None:
            step_ids.append(step.get("step_id"))
    if step_ids and len(step_ids) != len(set(step_ids)):
        errs.append(err("steps.step_id", "step_id должны быть уникальными.", "Проверьте дубликаты step_id."))

    for i, step in enumerate(steps if isinstance(steps, list) else [], start=1):
        si = i - 1
        if not isinstance(step, dict):
            errs.append(err(f"steps[{i}]", "Шаг должен быть словарём (YAML mapping).", "Проверьте структуру шага.", step_index=si))
            continue

        if step.get("step_id") is None:
            errs.append(err(f"steps[{i}].step_id", "step_id обязателен.", "Укажите числовой step_id (1,2,3...).", step_index=si))

        if not str(step.get("claim") or "").strip():
            errs.append(err(f"steps[{i}].claim", "claim обязателен.", "Коротко сформулируйте утверждение шага.", step_index=si))

        sources = _normalize_sources(step)
        if not isinstance(sources, list) or len(sources) == 0:
            errs.append(err(f"steps[{i}].sources", "Нужно добавить минимум 1 источник (sources).", "Добавьте хотя бы один source типа text/image/table.", step_index=si))
        else:
            for j, src in enumerate(sources, start=1):
                if not isinstance(src, dict):
                    errs.append(err(f"steps[{i}].sources[{j}]", "Источник должен быть словарём.", "Проверьте структуру sources.", step_index=si))
                    continue
                t = str(src.get("type") or "").strip().lower()
                if t == "figure":
                    t = "image"
                if t not in ("text", "image", "table"):
                    errs.append(err(f"steps[{i}].sources[{j}].type", "type должен быть text/image/table.", "Используйте: text, image или table.", step_index=si))
                if not str(src.get("source") or "").strip():
                    errs.append(err(f"steps[{i}].sources[{j}].source", "source обязателен.", "Пример: doi:10.... или arxiv:....", step_index=si))
                p = str(src.get("page") or "").strip()
                if p and p.lower() != "unknown" and not re.fullmatch(r"\d+", p):
                    errs.append(err(f"steps[{i}].sources[{j}].page", "page должен быть числом или 'unknown' (или пустым).", "Если номер страницы не применим/неизвестен — оставьте поле пустым.", step_index=si))
                if not str(src.get("snippet_or_summary") or "").strip():
                    errs.append(err(f"steps[{i}].sources[{j}].snippet_or_summary", "snippet_or_summary обязателен.", "Вставьте цитату/выжимку из источника.", step_index=si))
                if t in ("image", "table") and not str(src.get("locator") or "").strip():
                    errs.append(err(f"steps[{i}].sources[{j}].locator", "Для image/table желательно указать locator.", "Пример: Figure 3 / Table 2.", step_index=si))

        cond = step.get("conditions") or {}
        if not isinstance(cond, dict) or not cond:
            errs.append(err(f"steps[{i}].conditions", "conditions обязателен.", "Заполните условия (если в статье не указано — пишите 'unknown').", step_index=si))
        else:
            for k in (domain_required or []):
                if not str(cond.get(k) or "").strip():
                    errs.append(err(
                        f"steps[{i}].conditions.{k}",
                        f"conditions.{k} обязателен для выбранного домена.",
                        "Если в статье не указано — напишите 'unknown'.",
                        step_index=si,
                    ))

        if not str(step.get("inference") or "").strip():
            errs.append(err(f"steps[{i}].inference", "inference обязателен.", "Опишите логический вывод из sources+conditions.", step_index=si))
        if not str(step.get("next_question") or "").strip():
            errs.append(err(f"steps[{i}].next_question", "next_question обязателен.", "Сформулируйте вопрос для следующего шага.", step_index=si))

    # edges validation (optional)
    edges = doc.get("edges", None)
    if edges is not None:
        if not isinstance(edges, list):
            errs.append(err("edges", "edges должен быть списком пар.", "Пример: - [1, 2]"))
        else:
            sid_set = set([s.get("step_id") for s in steps if isinstance(s, dict)])
            for k, e in enumerate(edges, start=1):
                if not (isinstance(e, (list, tuple)) and len(e) == 2):
                    errs.append(err(f"edges[{k}]", "Ребро должно быть парой [from, to].", "Пример: [1, 2]."))
                    continue
                a, b = e[0], e[1]
                if sid_set and (a not in sid_set or b not in sid_set):
                    errs.append(err(f"edges[{k}]", "Ребро ссылается на несуществующий step_id.", "Проверьте номера шагов в edges."))

    return errs


def _format_errors_for_humans(errs):
    lines = []
    for e in (errs or []):
        if isinstance(e, str):
            lines.append(e)
            continue
        path = e.get("path") or "?"
        msg = e.get("message") or ""
        hint = e.get("hint") or ""
        if hint:
            lines.append(f"{path}: {msg}  👉 {hint}")
        else:
            lines.append(f"{path}: {msg}")
    return lines

print("Ок. Теперь запускайте ячейку: «ФОРМА».")

# -----------------------------
# Автономная HTML-форма для локального заполнения
# -----------------------------

from typing import Any

import json
from pathlib import Path
from typing import Any


def _default_state() -> dict[str, Any]:
    return {
        "_schema": "trajectory_form_state_v3",
        "expert": {"last_name": "", "first_name": "", "patronymic": "-"},
        "topic": "",
        "cutoff_year": 2020,
        "domain_query": "",
        "domain_language": "ru",
        "domain_qid": "Q336",
        "papers": [{"id": "", "year": 0, "title": ""}],
        "steps": [{
            "step_id": 1,
            "claim": "",
            "sources": [{"type": "text", "source": "", "page": "", "locator": "", "snippet_or_summary": ""}],
            "conditions": {"system": "unknown", "environment": "unknown", "protocol": "unknown", "notes": ""},
            "inference": "",
            "next_question": "",
        }],
        "edges": [],
        "selected_step_index": 0,
        "filename": "",
        "github": {"repo": "", "branch": "main", "path": "", "message": ""},
    }


_HTML_TEMPLATE = r'''<!doctype html>
<html lang="ru">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>__PAGE_TITLE__</title>
  <style>
    :root {
      --bg: #f8fafc;
      --card: #ffffff;
      --border: #d0d7de;
      --text: #0f172a;
      --muted: #475569;
      --accent: #2563eb;
      --accent-soft: #dbeafe;
      --success: #166534;
      --success-soft: #dcfce7;
      --warning: #92400e;
      --warning-soft: #fef3c7;
      --danger: #b91c1c;
      --danger-soft: #fee2e2;
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font-family: Inter, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: var(--text);
      background: linear-gradient(180deg, #eff6ff 0%, var(--bg) 180px);
    }
    .container { max-width: 1200px; margin: 0 auto; padding: 24px; }
    .hero {
      background: linear-gradient(135deg, rgba(37,99,235,0.12), rgba(37,99,235,0.03));
      border: 1px solid rgba(37,99,235,0.16);
      border-radius: 20px;
      padding: 24px;
      margin-bottom: 18px;
      box-shadow: 0 20px 50px rgba(15, 23, 42, 0.08);
    }
    .hero h1 { margin: 0 0 8px; font-size: 28px; }
    .hero p { margin: 0; color: var(--muted); line-height: 1.5; }
    .status {
      margin-top: 12px;
      display: inline-flex;
      gap: 8px;
      align-items: center;
      padding: 10px 12px;
      border-radius: 999px;
      background: rgba(255,255,255,0.7);
      border: 1px solid rgba(148, 163, 184, 0.35);
      color: var(--muted);
      font-size: 14px;
    }
    .toolbar, .toolbar-secondary {
      display: flex;
      gap: 10px;
      flex-wrap: wrap;
      margin-bottom: 14px;
    }
    button, .button-like {
      border: 0;
      border-radius: 12px;
      padding: 11px 16px;
      cursor: pointer;
      font-weight: 600;
      transition: transform .12s ease, box-shadow .12s ease, opacity .12s ease;
      box-shadow: 0 8px 18px rgba(15, 23, 42, 0.08);
      background: #ffffff;
      color: var(--text);
      border: 1px solid rgba(148, 163, 184, 0.28);
    }
    button:hover, .button-like:hover { transform: translateY(-1px); }
    button.primary { background: var(--accent); color: #fff; border-color: var(--accent); }
    button.success { background: var(--success); color: #fff; border-color: var(--success); }
    button.warning { background: #b45309; color: #fff; border-color: #b45309; }
    button.danger { background: var(--danger); color: #fff; border-color: var(--danger); }
    button.ghost { background: #fff; color: var(--muted); }
    input[type=file] { display: none; }
    .card {
      background: var(--card);
      border: 1px solid rgba(148, 163, 184, 0.25);
      border-radius: 18px;
      padding: 18px;
      margin-bottom: 16px;
      box-shadow: 0 14px 35px rgba(15, 23, 42, 0.05);
    }
    .card h2, .card h3, .card h4 { margin-top: 0; }
    .section-head {
      display: flex;
      justify-content: space-between;
      gap: 12px;
      align-items: center;
      margin-bottom: 12px;
    }
    .section-head p { margin: 4px 0 0; color: var(--muted); }
    .grid { display: grid; gap: 12px; }
    .grid.cols-2 { grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); }
    .grid.cols-3 { grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); }
    label.field { display: flex; flex-direction: column; gap: 6px; font-weight: 600; font-size: 14px; }
    label.field span.meta { color: var(--muted); font-weight: 500; }
    input[type=text], input[type=number], select, textarea {
      width: 100%;
      border-radius: 12px;
      border: 1px solid rgba(148, 163, 184, 0.4);
      padding: 11px 12px;
      font: inherit;
      color: var(--text);
      background: #fff;
    }
    textarea { min-height: 96px; resize: vertical; }
    input[readonly] { background: #f8fafc; }
    .muted { color: var(--muted); }
    .hint { color: var(--muted); font-size: 13px; }
    .pill {
      display: inline-flex;
      align-items: center;
      gap: 6px;
      padding: 6px 10px;
      border-radius: 999px;
      font-size: 12px;
      font-weight: 700;
    }
    .pill.info { background: #e0f2fe; color: #075985; }
    .pill.success { background: var(--success-soft); color: var(--success); }
    .pill.warning { background: var(--warning-soft); color: var(--warning); }
    .callout {
      border-radius: 14px;
      padding: 12px 14px;
      margin-top: 10px;
      line-height: 1.45;
      border: 1px solid transparent;
    }
    .callout.info { background: #eff6ff; color: #1d4ed8; border-color: #bfdbfe; }
    .callout.success { background: var(--success-soft); color: var(--success); border-color: #86efac; }
    .callout.warning { background: var(--warning-soft); color: var(--warning); border-color: #fcd34d; }
    .callout.error { background: var(--danger-soft); color: var(--danger); border-color: #fca5a5; }
    .paper-row, .source-card, .step-card {
      border: 1px solid rgba(148, 163, 184, 0.28);
      border-radius: 16px;
      padding: 14px;
      background: linear-gradient(180deg, rgba(255,255,255,0.98), rgba(248,250,252,0.98));
      margin-bottom: 12px;
    }
    .step-card { padding: 16px; }
    .step-header, .subsection-header {
      display: flex;
      justify-content: space-between;
      gap: 10px;
      align-items: center;
      margin-bottom: 10px;
    }
    .step-title { font-size: 18px; font-weight: 700; }
    .step-claim-preview { color: var(--muted); font-size: 13px; }
    .mini-actions { display: flex; flex-wrap: wrap; gap: 8px; }
    .mini-actions button { padding: 8px 11px; border-radius: 10px; font-size: 13px; }
    .edge-table-wrap { overflow-x: auto; }
    table.edge-table {
      width: 100%;
      border-collapse: separate;
      border-spacing: 0;
      min-width: 720px;
    }
    .edge-table th, .edge-table td {
      border-bottom: 1px solid rgba(148,163,184,.2);
      padding: 10px 12px;
      vertical-align: top;
      background: white;
    }
    .edge-table th {
      position: sticky;
      top: 0;
      background: #f8fafc;
      z-index: 1;
      font-size: 13px;
      text-align: left;
    }
    .edge-table td.center { text-align: center; }
    .edge-check { width: 18px; height: 18px; }
    .results-list { display: grid; gap: 8px; margin-top: 10px; }
    .result-option {
      padding: 10px 12px;
      border-radius: 12px;
      border: 1px solid rgba(148,163,184,.25);
      background: #fff;
      cursor: pointer;
      text-align: left;
    }
    .result-option:hover { border-color: #93c5fd; box-shadow: 0 6px 15px rgba(37,99,235,.08); }
    .result-option strong { display: block; }
    .footer-note { color: var(--muted); font-size: 13px; margin-top: 10px; }
    .hidden { display: none !important; }
    code {
      padding: 2px 5px;
      border-radius: 6px;
      background: #e2e8f0;
      font-size: 0.95em;
    }
    @media (max-width: 760px) {
      .container { padding: 16px; }
      .hero { padding: 18px; }
      .section-head { flex-direction: column; align-items: flex-start; }
    }
  </style>
</head>
<body>
  <div class="container">
    <section class="hero">
      <h1>Task 1 — автономная форма Reasoning Trajectories</h1>
      <p>Эту HTML-форму можно скачать из ноутбука и заполнять локально на устройстве. Она хранит черновик в браузере, умеет экспортировать текущую форму в <code>.yaml</code> и выгружать / загружать черновик как <code>.json</code>.</p>
      <div class="status" id="autosaveStatus">Черновик ещё не сохранён</div>
    </section>

    <section class="toolbar">
      <button class="primary" id="exportYamlBtn">Скачать YAML</button>
      <button class="success" id="saveDraftBtn">Скачать черновик JSON</button>
      <label class="button-like" for="loadDraftInput">Загрузить черновик JSON</label>
      <input type="file" id="loadDraftInput" accept=".json,application/json">
      <button class="ghost" id="restoreBtn">Восстановить из автосохранения</button>
      <button class="warning" id="clearDraftBtn">Сбросить автосохранение</button>
    </section>

    <section class="card">
      <div class="section-head">
        <div>
          <h2>Эксперт и основная информация</h2>
          <p>Эти поля нужны для формирования <code>submission_id</code> и итогового артефакта.</p>
        </div>
        <span class="pill info">artifact v2</span>
      </div>
      <div class="grid cols-3">
        <label class="field"><span>Фамилия*</span><input type="text" id="expertLast" placeholder="Иванов"></label>
        <label class="field"><span>Имя*</span><input type="text" id="expertFirst" placeholder="Иван"></label>
        <label class="field"><span>Отчество*</span><input type="text" id="expertPat" placeholder="Иванович или -"></label>
      </div>
      <div class="grid cols-2" style="margin-top:12px;">
        <label class="field"><span>Topic*</span><input type="text" id="topicInput" placeholder="Коротко: о чём траектория"></label>
        <label class="field"><span>Cutoff year*</span><input type="number" id="cutoffYearInput" min="1800" max="2100" step="1"></label>
      </div>
      <div class="grid cols-2" style="margin-top:12px;">
        <label class="field"><span>filename</span><input type="text" id="filenameInput" placeholder="Оставьте пустым — имя сформируется автоматически"></label>
        <label class="field"><span>Предпросмотр ID</span><input type="text" id="submissionPreview" readonly></label>
      </div>
    </section>

    <section class="card">
      <div class="section-head">
        <div>
          <h2>Домен (Wikidata QID)</h2>
          <p>Можно ввести QID вручную или выполнить поиск через Wikidata прямо из формы.</p>
        </div>
        <span class="pill success" id="domainBadge">domain ready</span>
      </div>
      <div class="grid cols-3">
        <label class="field" style="grid-column: span 2;"><span>Wikidata search</span><input type="text" id="domainQuery" placeholder="Science / Наука / Lithium-ion battery"></label>
        <label class="field"><span>Lang</span><select id="domainLanguage"><option value="ru">Русский (ru)</option><option value="en">English (en)</option></select></label>
      </div>
      <div class="toolbar-secondary" style="margin-top:12px; margin-bottom:0;">
        <button class="ghost" id="domainSearchBtn">Найти в Wikidata</button>
      </div>
      <div class="results-list" id="wikidataResults"></div>
      <div class="grid cols-2" style="margin-top:12px;">
        <label class="field"><span>Domain (QID)*</span><input type="text" id="domainQid" placeholder="Q336"></label>
        <label class="field"><span>Обязательные conditions для выбранного домена</span><input type="text" id="requiredConditions" readonly></label>
      </div>
      <div class="callout info" id="wikidataStatus">По умолчанию используется домен <code>Q336</code>, если ничего не выбрано.</div>
    </section>

    <section class="card">
      <div class="section-head">
        <div>
          <h2>Публикации</h2>
          <p>Для каждой публикации заполните <code>id</code>, <code>year</code> и <code>title</code>.</p>
        </div>
        <div class="mini-actions">
          <button class="ghost" id="addPaperBtn">Добавить публикацию</button>
        </div>
      </div>
      <div id="papersContainer"></div>
    </section>

    <section class="card">
      <div class="section-head">
        <div>
          <h2>Шаги reasoning trajectory</h2>
          <p>Каждый шаг включает claim, набор источников, conditions, inference и next question.</p>
        </div>
        <div class="mini-actions">
          <button class="ghost" id="addStepBtn">Добавить шаг</button>
        </div>
      </div>
      <div id="stepsContainer"></div>
    </section>

    <section class="card">
      <div class="section-head">
        <div>
          <h2>Связи между шагами</h2>
          <p>Отмечайте направленные связи между шагами; петли <code>i → i</code> недоступны.</p>
        </div>
        <span class="pill warning" id="edgeCountPill">0 edges</span>
      </div>
      <div class="edge-table-wrap" id="edgesMatrixWrap"></div>
    </section>

    <section class="card">
      <div class="section-head">
        <div>
          <h2>Что именно сохраняется</h2>
          <p>Автосохранение хранится в <code>localStorage</code> браузера, а финальный YAML скачивается как локальный файл.</p>
        </div>
      </div>
      <div class="callout info">Загрузка / выгрузка HTML-файла работает через браузерные <code>Blob</code>-объекты и ссылку из <code>URL.createObjectURL()</code>; черновики формы сохраняются в <code>localStorage</code>; импорт JSON-черновика читается через <code>FileReader</code>.</div>
      <div class="footer-note">Совет: после завершения разметки скачайте итоговый YAML и при необходимости отдельно сохраните JSON-черновик как резервную копию.</div>
    </section>

    <div id="globalMessage"></div>
  </div>

  <script>
  const INITIAL_STATE = __INITIAL_STATE__;
  const DOMAIN_CONFIGS = __DOMAIN_CONFIGS__;
  const STORAGE_KEY = '__STORAGE_KEY__';
  const FORM_SCHEMA = 'trajectory_form_state_v3';
  const DEFAULT_DOMAIN_QID = (DOMAIN_CONFIGS.find((x) => x.wikidata_qid) || {}).wikidata_qid || 'Q336';

  const dom = {
    autosaveStatus: document.getElementById('autosaveStatus'),
    globalMessage: document.getElementById('globalMessage'),
    exportYamlBtn: document.getElementById('exportYamlBtn'),
    saveDraftBtn: document.getElementById('saveDraftBtn'),
    loadDraftInput: document.getElementById('loadDraftInput'),
    restoreBtn: document.getElementById('restoreBtn'),
    clearDraftBtn: document.getElementById('clearDraftBtn'),
    expertLast: document.getElementById('expertLast'),
    expertFirst: document.getElementById('expertFirst'),
    expertPat: document.getElementById('expertPat'),
    topicInput: document.getElementById('topicInput'),
    cutoffYearInput: document.getElementById('cutoffYearInput'),
    filenameInput: document.getElementById('filenameInput'),
    submissionPreview: document.getElementById('submissionPreview'),
    domainQuery: document.getElementById('domainQuery'),
    domainLanguage: document.getElementById('domainLanguage'),
    domainSearchBtn: document.getElementById('domainSearchBtn'),
    wikidataResults: document.getElementById('wikidataResults'),
    wikidataStatus: document.getElementById('wikidataStatus'),
    domainQid: document.getElementById('domainQid'),
    requiredConditions: document.getElementById('requiredConditions'),
    domainBadge: document.getElementById('domainBadge'),
    papersContainer: document.getElementById('papersContainer'),
    addPaperBtn: document.getElementById('addPaperBtn'),
    stepsContainer: document.getElementById('stepsContainer'),
    addStepBtn: document.getElementById('addStepBtn'),
    edgesMatrixWrap: document.getElementById('edgesMatrixWrap'),
    edgeCountPill: document.getElementById('edgeCountPill'),
  };

  const CYR = {
    'а':'a','б':'b','в':'v','г':'g','д':'d','е':'e','ё':'e','ж':'zh','з':'z','и':'i','й':'i','к':'k','л':'l','м':'m','н':'n','о':'o','п':'p','р':'r','с':'s','т':'t','у':'u','ф':'f','х':'kh','ц':'ts','ч':'ch','ш':'sh','щ':'shch','ы':'y','э':'e','ю':'yu','я':'ya','ь':'','ъ':''
  };

  function deepClone(value) {
    return JSON.parse(JSON.stringify(value));
  }

  function defaultSource() {
    return { type: 'text', source: '', page: '', locator: '', snippet_or_summary: '' };
  }

  function defaultStep(index) {
    return {
      step_id: index,
      claim: '',
      sources: [defaultSource()],
      conditions: { system: 'unknown', environment: 'unknown', protocol: 'unknown', notes: '' },
      inference: '',
      next_question: '',
    };
  }

  function defaultPaper() {
    return { id: '', year: 0, title: '' };
  }

  function defaultState() {
    return {
      _schema: FORM_SCHEMA,
      expert: { last_name: '', first_name: '', patronymic: '-' },
      topic: '',
      cutoff_year: 2020,
      domain_query: '',
      domain_language: 'ru',
      domain_qid: DEFAULT_DOMAIN_QID,
      papers: [defaultPaper()],
      steps: [defaultStep(1)],
      edges: [],
      selected_step_index: 0,
      filename: '',
      github: { repo: '', branch: 'main', path: '', message: '' },
    };
  }

  function slugify(text) {
    const prepared = transliterate(text || '').toLowerCase();
    return prepared.replace(/[^a-z0-9]+/g, '_').replace(/_+/g, '_').replace(/^_+|_+$/g, '').slice(0, 60) || 'trajectory';
  }

  function transliterate(text) {
    const src = String(text || '');
    let out = '';
    for (const ch of src) {
      const low = ch.toLowerCase();
      let repl = CYR[low];
      if (typeof repl === 'undefined') {
        out += ch;
        continue;
      }
      if (ch !== low && repl) {
        repl = repl.charAt(0).toUpperCase() + repl.slice(1);
      }
      out += repl;
    }
    return out;
  }

  function cleanTextPreview(text, limit = 48) {
    const value = String(text || '').replace(/\s+/g, ' ').trim();
    if (value.length <= limit) return value;
    return value.slice(0, Math.max(1, limit - 1)).trimEnd() + '…';
  }

  function nowUtc() {
    return new Date().toISOString().replace(/\.\d{3}Z$/, 'Z');
  }

  function domainConfigFor(qid) {
    return DOMAIN_CONFIGS.find((item) => String(item.wikidata_qid || '').trim() === String(qid || '').trim()) || null;
  }

  function requiredConditionKeys(qid) {
    const cfg = domainConfigFor(qid);
    const keys = (cfg && Array.isArray(cfg.required_conditions)) ? cfg.required_conditions.slice() : [];
    return keys;
  }

  function normalizeState(raw) {
    const base = defaultState();
    const state = Object.assign({}, base, raw || {});
    state.expert = Object.assign({}, base.expert, state.expert || {});
    state.github = Object.assign({}, base.github, state.github || {});
    state.domain_language = ['ru', 'en'].includes(state.domain_language) ? state.domain_language : 'ru';
    state.domain_qid = String(state.domain_qid || DEFAULT_DOMAIN_QID || 'Q336').trim();
    state.papers = Array.isArray(state.papers) && state.papers.length ? state.papers.map((paper) => ({
      id: String((paper || {}).id || ''),
      year: Number((paper || {}).year || 0),
      title: String((paper || {}).title || ''),
    })) : [defaultPaper()];
    state.steps = Array.isArray(state.steps) && state.steps.length ? state.steps.map((step, idx) => normalizeStep(step, idx + 1, state.domain_qid)) : [defaultStep(1)];
    state.edges = normalizeEdges(state.edges, state.steps.length);
    state.selected_step_index = Number.isInteger(state.selected_step_index) ? state.selected_step_index : 0;
    state._schema = FORM_SCHEMA;
    return state;
  }

  function normalizeStep(step, index, domainQid) {
    const base = defaultStep(index);
    const normalized = Object.assign({}, base, step || {});
    normalized.step_id = index;
    normalized.claim = String(normalized.claim || '');
    normalized.inference = String(normalized.inference || '');
    normalized.next_question = String(normalized.next_question || '');
    normalized.sources = Array.isArray(normalized.sources) && normalized.sources.length ? normalized.sources.map((src) => ({
      type: ['text', 'image', 'table'].includes(String((src || {}).type || '').trim()) ? String(src.type).trim() : 'text',
      source: String((src || {}).source || ''),
      page: String((src || {}).page || ''),
      locator: String((src || {}).locator || ''),
      snippet_or_summary: String((src || {}).snippet_or_summary || ''),
    })) : [defaultSource()];
    normalized.conditions = Object.assign({}, base.conditions, normalized.conditions || {});
    for (const key of requiredConditionKeys(domainQid)) {
      if (!(key in normalized.conditions)) normalized.conditions[key] = '';
    }
    return normalized;
  }

  function normalizeEdges(edges, stepCount) {
    const out = [];
    const seen = new Set();
    if (!Array.isArray(edges)) return out;
    for (const pair of edges) {
      if (!Array.isArray(pair) || pair.length < 2) continue;
      const from = Number(pair[0]);
      const to = Number(pair[1]);
      if (!Number.isFinite(from) || !Number.isFinite(to) || from === to) continue;
      if (from < 1 || to < 1 || from > stepCount || to > stepCount) continue;
      const key = `${from}->${to}`;
      if (seen.has(key)) continue;
      seen.add(key);
      out.push([from, to]);
    }
    return out;
  }

  let state = normalizeState(INITIAL_STATE);

  function syncTopFieldsToState() {
    state.expert.last_name = dom.expertLast.value;
    state.expert.first_name = dom.expertFirst.value;
    state.expert.patronymic = dom.expertPat.value;
    state.topic = dom.topicInput.value;
    state.cutoff_year = Number(dom.cutoffYearInput.value || 0);
    state.filename = dom.filenameInput.value;
    state.domain_query = dom.domainQuery.value;
    state.domain_language = dom.domainLanguage.value;
    state.domain_qid = String(dom.domainQid.value || DEFAULT_DOMAIN_QID || 'Q336').trim();
  }

  function syncStateToTopFields() {
    dom.expertLast.value = state.expert.last_name || '';
    dom.expertFirst.value = state.expert.first_name || '';
    dom.expertPat.value = state.expert.patronymic || '-';
    dom.topicInput.value = state.topic || '';
    dom.cutoffYearInput.value = String(state.cutoff_year || 2020);
    dom.filenameInput.value = state.filename || '';
    dom.domainQuery.value = state.domain_query || '';
    dom.domainLanguage.value = state.domain_language || 'ru';
    dom.domainQid.value = state.domain_qid || DEFAULT_DOMAIN_QID || 'Q336';
  }

  function renderPapers() {
    dom.papersContainer.innerHTML = '';
    state.papers.forEach((paper, index) => {
      const row = document.createElement('div');
      row.className = 'paper-row';
      row.innerHTML = `
        <div class="subsection-header">
          <div><strong>Публикация ${index + 1}</strong></div>
          <div class="mini-actions">
            <button class="danger" data-remove-paper="${index}">Удалить</button>
          </div>
        </div>
        <div class="grid cols-3">
          <label class="field"><span>id*</span><input type="text" data-paper-field="id" data-paper-index="${index}" value="${escapeAttr(paper.id)}" placeholder="doi:... / arxiv:... / openalex:... / url"></label>
          <label class="field"><span>year*</span><input type="number" min="0" step="1" data-paper-field="year" data-paper-index="${index}" value="${escapeAttr(paper.year || '')}"></label>
          <label class="field"><span>title*</span><input type="text" data-paper-field="title" data-paper-index="${index}" value="${escapeAttr(paper.title)}" placeholder="Название статьи"></label>
        </div>`;
      dom.papersContainer.appendChild(row);
    });
  }

  function renderSteps() {
    dom.stepsContainer.innerHTML = '';
    state.steps.forEach((step, index) => {
      const requiredKeys = requiredConditionKeys(state.domain_qid);
      const conditionKeys = ['system', 'environment', 'protocol', 'notes', ...requiredKeys.filter((key) => !['system', 'environment', 'protocol', 'notes'].includes(key))];
      const wrap = document.createElement('div');
      wrap.className = 'step-card';
      const sourcesHtml = step.sources.map((source, sourceIndex) => `
        <div class="source-card">
          <div class="subsection-header">
            <div><strong>Источник ${sourceIndex + 1}</strong></div>
            <div class="mini-actions"><button class="danger" data-remove-source="${index}:${sourceIndex}">Удалить источник</button></div>
          </div>
          <div class="grid cols-3">
            <label class="field"><span>type</span>
              <select data-source-field="type" data-step-index="${index}" data-source-index="${sourceIndex}">
                <option value="text" ${source.type === 'text' ? 'selected' : ''}>Text</option>
                <option value="image" ${source.type === 'image' ? 'selected' : ''}>Image/Figure</option>
                <option value="table" ${source.type === 'table' ? 'selected' : ''}>Table</option>
              </select>
            </label>
            <label class="field"><span>page</span><input type="text" data-source-field="page" data-step-index="${index}" data-source-index="${sourceIndex}" value="${escapeAttr(source.page)}" placeholder="например, 1"></label>
            <label class="field"><span>locator</span><input type="text" data-source-field="locator" data-step-index="${index}" data-source-index="${sourceIndex}" value="${escapeAttr(source.locator)}" placeholder="Figure 3 / Table 2"></label>
          </div>
          <label class="field" style="margin-top:10px;"><span>source*</span><input type="text" data-source-field="source" data-step-index="${index}" data-source-index="${sourceIndex}" value="${escapeAttr(source.source)}" placeholder="doi:... / arxiv:... / openalex:... / url"></label>
          <label class="field" style="margin-top:10px;"><span>snippet*</span><textarea data-source-field="snippet_or_summary" data-step-index="${index}" data-source-index="${sourceIndex}" placeholder="Цитата/выжимка/описание">${escapeHtml(source.snippet_or_summary)}</textarea></label>
        </div>`).join('');
      const conditionsHtml = conditionKeys.map((key) => `
        <label class="field"><span>${escapeHtml(key)}${['system','environment','protocol'].includes(key) ? '*' : ''}</span><input type="text" data-condition-key="${escapeAttr(key)}" data-step-index="${index}" value="${escapeAttr((step.conditions || {})[key] || '')}" placeholder="${key === 'notes' ? '' : 'unknown'}"></label>`).join('');
      wrap.innerHTML = `
        <div class="step-header">
          <div>
            <div class="step-title">Шаг ${index + 1}</div>
            <div class="step-claim-preview">${escapeHtml(cleanTextPreview(step.claim || 'Утверждение пока не заполнено', 90))}</div>
          </div>
          <div class="mini-actions">
            <button class="ghost" data-add-source="${index}">Добавить источник</button>
            <button class="danger" data-remove-step="${index}">Удалить шаг</button>
          </div>
        </div>
        <label class="field"><span>claim*</span><textarea data-step-field="claim" data-step-index="${index}" placeholder="Утверждение шага">${escapeHtml(step.claim)}</textarea></label>
        <div style="margin-top:12px;">
          <div class="subsection-header"><div><strong>Sources</strong></div></div>
          ${sourcesHtml}
        </div>
        <div style="margin-top:12px;">
          <div class="subsection-header"><div><strong>Conditions</strong></div><div class="hint">Если в статье не указано — пишите <code>unknown</code>.</div></div>
          <div class="grid cols-2">${conditionsHtml}</div>
        </div>
        <div class="grid cols-2" style="margin-top:12px;">
          <label class="field"><span>inference*</span><textarea data-step-field="inference" data-step-index="${index}" placeholder="Логический вывод">${escapeHtml(step.inference)}</textarea></label>
          <label class="field"><span>next_question*</span><textarea data-step-field="next_question" data-step-index="${index}" placeholder="Вопрос для следующего шага">${escapeHtml(step.next_question)}</textarea></label>
        </div>`;
      dom.stepsContainer.appendChild(wrap);
    });
  }

  function renderEdgesMatrix() {
    const rows = [];
    const stepCount = state.steps.length;
    if (!stepCount) {
      dom.edgesMatrixWrap.innerHTML = '<div class="hint">Сначала добавьте хотя бы один шаг.</div>';
      dom.edgeCountPill.textContent = '0 edges';
      return;
    }
    const headers = state.steps.map((step, index) => `<th>${index + 1}: ${escapeHtml(cleanTextPreview(step.claim || '', 32) || 'шаг')}</th>`).join('');
    for (let from = 1; from <= stepCount; from += 1) {
      const cells = [];
      for (let to = 1; to <= stepCount; to += 1) {
        if (from === to) {
          cells.push('<td class="center">—</td>');
          continue;
        }
        const checked = state.edges.some((pair) => pair[0] === from && pair[1] === to);
        cells.push(`<td class="center"><input class="edge-check" type="checkbox" data-edge-from="${from}" data-edge-to="${to}" ${checked ? 'checked' : ''}></td>`);
      }
      rows.push(`<tr><th>${from}: ${escapeHtml(cleanTextPreview(state.steps[from - 1].claim || '', 38) || 'шаг')}</th>${cells.join('')}</tr>`);
    }
    dom.edgesMatrixWrap.innerHTML = `<table class="edge-table"><thead><tr><th>from \\ to</th>${headers}</tr></thead><tbody>${rows.join('')}</tbody></table>`;
    dom.edgeCountPill.textContent = `${state.edges.length} edges`;
  }

  function refreshMeta() {
    const expertSlug = slugify([state.expert.last_name, state.expert.first_name, state.expert.patronymic].filter(Boolean).join(' '));
    dom.submissionPreview.value = `${expertSlug}__pending_hash`;
    const required = requiredConditionKeys(state.domain_qid);
    dom.requiredConditions.value = required.length ? required.join(', ') : 'только базовые поля system/environment/protocol/notes';
    const cfg = domainConfigFor(state.domain_qid);
    dom.domainBadge.textContent = cfg ? `${cfg.title || cfg.domain_id || state.domain_qid}` : `Custom domain ${state.domain_qid || DEFAULT_DOMAIN_QID}`;
  }

  function renderAll(save = false) {
    syncStateToTopFields();
    renderPapers();
    renderSteps();
    renderEdgesMatrix();
    refreshMeta();
    if (save) saveAutosave('render');
  }

  function setMessage(kind, html) {
    if (!html) {
      dom.globalMessage.innerHTML = '';
      return;
    }
    dom.globalMessage.innerHTML = `<div class="callout ${kind}" style="margin-bottom:18px;">${html}</div>`;
  }

  function setAutosaveStatus(text, kind = 'info') {
    dom.autosaveStatus.textContent = text;
    dom.autosaveStatus.style.color = kind === 'error' ? '#b91c1c' : kind === 'success' ? '#166534' : '#475569';
  }

  function saveAutosave(reason = 'autosave') {
    try {
      syncTopFieldsToState();
      const payload = deepClone(state);
      payload._schema = FORM_SCHEMA;
      payload._saved_at = nowUtc();
      payload._saved_reason = reason;
      localStorage.setItem(STORAGE_KEY, JSON.stringify(payload));
      setAutosaveStatus(`Черновик автосохранён (${payload._saved_at})`, 'success');
    } catch (err) {
      setAutosaveStatus(`Не удалось сохранить черновик: ${err}`, 'error');
    }
  }

  function restoreAutosave(showMessage = true) {
    try {
      const raw = localStorage.getItem(STORAGE_KEY);
      if (!raw) {
        if (showMessage) setMessage('warning', 'Автосохранённый черновик пока не найден.');
        return false;
      }
      state = normalizeState(JSON.parse(raw));
      renderAll(false);
      setAutosaveStatus(`Черновик восстановлен (${state._saved_at || 'без даты'})`, 'success');
      if (showMessage) setMessage('success', 'Форма восстановлена из локального автосохранения браузера.');
      return true;
    } catch (err) {
      setMessage('error', `Не удалось восстановить черновик: ${escapeHtml(String(err))}`);
      return false;
    }
  }

  function addPaper() {
    state.papers.push(defaultPaper());
    renderAll(true);
  }

  function removePaper(index) {
    state.papers.splice(index, 1);
    if (!state.papers.length) state.papers.push(defaultPaper());
    renderAll(true);
  }

  function addStep() {
    state.steps.push(defaultStep(state.steps.length + 1));
    state.steps = state.steps.map((step, idx) => normalizeStep(step, idx + 1, state.domain_qid));
    state.edges = normalizeEdges(state.edges, state.steps.length);
    renderAll(true);
  }

  function removeStep(index) {
    state.steps.splice(index, 1);
    if (!state.steps.length) state.steps.push(defaultStep(1));
    state.steps = state.steps.map((step, idx) => normalizeStep(step, idx + 1, state.domain_qid));
    state.edges = normalizeEdges(state.edges.filter((pair) => pair[0] !== index + 1 && pair[1] !== index + 1).map((pair) => [pair[0] > index + 1 ? pair[0] - 1 : pair[0], pair[1] > index + 1 ? pair[1] - 1 : pair[1]]), state.steps.length);
    renderAll(true);
  }

  function addSource(stepIndex) {
    state.steps[stepIndex].sources.push(defaultSource());
    renderAll(true);
  }

  function removeSource(stepIndex, sourceIndex) {
    state.steps[stepIndex].sources.splice(sourceIndex, 1);
    if (!state.steps[stepIndex].sources.length) state.steps[stepIndex].sources.push(defaultSource());
    renderAll(true);
  }

  async function computeSubmissionMeta(doc) {
    const expertSlug = slugify([doc.expert.last_name, doc.expert.first_name, doc.expert.patronymic].filter(Boolean).join(' '));
    const canonical = JSON.stringify(doc);
    const bytes = new TextEncoder().encode(canonical);
    const hashBuffer = await crypto.subtle.digest('SHA-256', bytes);
    const hashArray = Array.from(new Uint8Array(hashBuffer));
    const fullHash = hashArray.map((b) => b.toString(16).padStart(2, '0')).join('');
    const shortHash = fullHash.slice(0, 12);
    return {
      artifact_hash: shortHash,
      submission_id: `${expertSlug || 'expert'}__${shortHash}`,
      latin_slug: expertSlug || 'expert',
      latin_full_name: transliterate([doc.expert.last_name, doc.expert.first_name, doc.expert.patronymic].filter(Boolean).join(' ')).trim(),
    };
  }

  function currentDocBase() {
    syncTopFieldsToState();
    state.steps = state.steps.map((step, idx) => normalizeStep(step, idx + 1, state.domain_qid));
    state.edges = normalizeEdges(state.edges, state.steps.length);
    return {
      artifact_version: 2,
      topic: String(state.topic || '').trim(),
      domain: String(state.domain_qid || DEFAULT_DOMAIN_QID || 'Q336').trim(),
      cutoff_year: Number(state.cutoff_year || 0),
      papers: state.papers.map((paper) => ({ id: String(paper.id || '').trim(), year: Number(paper.year || 0), title: String(paper.title || '').trim() })),
      steps: state.steps.map((step, idx) => ({
        step_id: idx + 1,
        claim: String(step.claim || '').trim(),
        sources: step.sources.map((src) => ({
          type: String(src.type || 'text').trim() || 'text',
          source: String(src.source || '').trim(),
          snippet_or_summary: String(src.snippet_or_summary || '').trim(),
          ...(String(src.page || '').trim() ? { page: String(src.page || '').trim() } : {}),
          ...(String(src.locator || '').trim() ? { locator: String(src.locator || '').trim() } : {}),
        })),
        conditions: Object.fromEntries(Object.entries(step.conditions || {}).map(([key, value]) => [String(key), String(value || '')])),
        inference: String(step.inference || '').trim(),
        next_question: String(step.next_question || '').trim(),
      })),
      edges: normalizeEdges(state.edges, state.steps.length),
      generated_at: nowUtc(),
      expert: {
        last_name: String(state.expert.last_name || '').trim(),
        first_name: String(state.expert.first_name || '').trim(),
        patronymic: String(state.expert.patronymic || '').trim(),
      },
    };
  }

  function validateDoc(doc) {
    const errors = [];
    if (!doc.expert.last_name || !doc.expert.first_name || !doc.expert.patronymic) errors.push('Заполните ФИО эксперта.');
    if (!doc.topic) errors.push('Заполните поле Topic.');
    if (!doc.domain) errors.push('Укажите Domain (QID).');
    if (!Number.isFinite(doc.cutoff_year) || doc.cutoff_year <= 0) errors.push('Cutoff year должен быть положительным числом.');
    const validPapers = doc.papers.filter((paper) => paper.id || paper.title || paper.year);
    if (!validPapers.length) errors.push('Добавьте хотя бы одну публикацию.');
    validPapers.forEach((paper, index) => {
      if (!paper.id || !paper.title || !paper.year) {
        errors.push(`papers[${index + 1}]: заполните id, year и title.`);
      }
    });
    doc.steps.forEach((step, index) => {
      if (!step.claim) errors.push(`steps[${index + 1}]: заполните claim.`);
      if (!step.inference) errors.push(`steps[${index + 1}]: заполните inference.`);
      if (!step.next_question) errors.push(`steps[${index + 1}]: заполните next_question.`);
      if (!Array.isArray(step.sources) || !step.sources.length) {
        errors.push(`steps[${index + 1}]: добавьте хотя бы один источник.`);
      }
      (step.sources || []).forEach((src, srcIndex) => {
        if (!src.source || !src.snippet_or_summary) errors.push(`steps[${index + 1}].sources[${srcIndex + 1}]: заполните source и snippet_or_summary.`);
      });
      ['system', 'environment', 'protocol'].forEach((key) => {
        if (!(key in (step.conditions || {}))) errors.push(`steps[${index + 1}].conditions.${key}: поле отсутствует.`);
      });
    });
    return errors;
  }

  function yamlScalar(value) {
    if (value === null || typeof value === 'undefined') return '""';
    if (typeof value === 'number') return Number.isFinite(value) ? String(value) : '0';
    if (typeof value === 'boolean') return value ? 'true' : 'false';
    const text = String(value);
    if (text === '') return '""';
    if (/^[A-Za-z0-9_:\/.+\-]+$/.test(text) && !['true', 'false', 'null'].includes(text.toLowerCase())) return text;
    return JSON.stringify(text);
  }

  function toYaml(value, indent = 0) {
    const pad = '  '.repeat(indent);
    if (Array.isArray(value)) {
      if (!value.length) return '[]';
      return value.map((item) => {
        if (item && typeof item === 'object') {
          const nested = toYaml(item, indent + 1);
          const lines = String(nested).split('\n');
          return `${pad}- ${lines[0]}\n${lines.slice(1).map((line) => `${pad}  ${line}`).join('\n')}`;
        }
        return `${pad}- ${yamlScalar(item)}`;
      }).join('\n');
    }
    if (value && typeof value === 'object') {
      const entries = Object.entries(value);
      if (!entries.length) return '{}';
      return entries.map(([key, item]) => {
        if (item && typeof item === 'object') {
          const nested = toYaml(item, indent + 1);
          if (Array.isArray(item)) return `${pad}${key}:\n${nested}`;
          return `${pad}${key}:\n${nested}`;
        }
        return `${pad}${key}: ${yamlScalar(item)}`;
      }).join('\n');
    }
    return `${pad}${yamlScalar(value)}`;
  }

  function downloadBlob(filename, blob) {
    const url = URL.createObjectURL(blob);
    const link = document.createElement('a');
    link.href = url;
    link.download = filename;
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(() => URL.revokeObjectURL(url), 1000);
  }

  async function exportYaml() {
    const doc = currentDocBase();
    const errors = validateDoc(doc);
    if (errors.length) {
      setMessage('error', `<strong>Форма заполнена не полностью.</strong><br>${errors.map((item) => escapeHtml(item)).join('<br>')}`);
      return;
    }
    const meta = await computeSubmissionMeta(doc);
    doc.expert.full_name = [doc.expert.last_name, doc.expert.first_name, doc.expert.patronymic].filter(Boolean).join(' ').trim();
    doc.expert.latin_full_name = meta.latin_full_name;
    doc.expert.latin_slug = meta.latin_slug;
    doc.artifact_hash = meta.artifact_hash;
    doc.submission_id = meta.submission_id;
    dom.submissionPreview.value = doc.submission_id;
    const yamlText = toYaml(doc) + '\n';
    const filename = (String(state.filename || '').trim() || `${doc.submission_id}.yaml`).replace(/\s+/g, '_');
    downloadBlob(filename.toLowerCase().endsWith('.yaml') || filename.toLowerCase().endsWith('.yml') ? filename : `${filename}.yaml`, new Blob([yamlText], { type: 'text/yaml;charset=utf-8' }));
    saveAutosave('export_yaml');
    setMessage('success', `YAML собран и скачан: <code>${escapeHtml(filename)}</code>`);
  }

  function saveDraftJson() {
    syncTopFieldsToState();
    const payload = deepClone(state);
    payload._schema = FORM_SCHEMA;
    payload._saved_at = nowUtc();
    payload._saved_reason = 'manual_json_export';
    const topicSlug = slugify(payload.topic || 'trajectory_form');
    downloadBlob(`${topicSlug || 'trajectory_form'}__draft.json`, new Blob([JSON.stringify(payload, null, 2)], { type: 'application/json;charset=utf-8' }));
    saveAutosave('manual_json_export');
    setMessage('success', 'JSON-черновик скачан на устройство.');
  }

  async function loadDraftFromFile(file) {
    if (!file) return;
    const reader = new FileReader();
    reader.onload = () => {
      try {
        const parsed = JSON.parse(String(reader.result || '{}'));
        state = normalizeState(parsed);
        renderAll(false);
        saveAutosave('import_json');
        setMessage('success', `Черновик <code>${escapeHtml(file.name)}</code> загружен в форму.`);
      } catch (err) {
        setMessage('error', `Не удалось прочитать JSON-черновик: ${escapeHtml(String(err))}`);
      } finally {
        dom.loadDraftInput.value = '';
      }
    };
    reader.readAsText(file);
  }

  async function searchWikidata() {
    const query = String(dom.domainQuery.value || '').trim();
    const language = String(dom.domainLanguage.value || 'ru').trim() || 'ru';
    if (!query) {
      setMessage('warning', 'Введите текст для поиска в Wikidata.');
      return;
    }
    dom.wikidataStatus.innerHTML = 'Ищу…';
    dom.wikidataResults.innerHTML = '';
    try {
      const params = new URLSearchParams({
        action: 'wbsearchentities',
        search: query,
        language,
        format: 'json',
        limit: '10',
        origin: '*',
      });
      const resp = await fetch(`https://www.wikidata.org/w/api.php?${params.toString()}`, { headers: { 'Accept': 'application/json' } });
      if (!resp.ok) throw new Error(`HTTP ${resp.status}`);
      const payload = await resp.json();
      const results = Array.isArray(payload.search) ? payload.search : [];
      if (!results.length) {
        dom.wikidataStatus.innerHTML = '<span class="muted">Ничего не найдено.</span>';
        return;
      }
      dom.wikidataStatus.innerHTML = `<span class="muted">Найдено: ${results.length}</span>`;
      results.forEach((item) => {
        const button = document.createElement('button');
        button.type = 'button';
        button.className = 'result-option';
        const qid = String(item.id || '').trim();
        button.innerHTML = `<strong>${escapeHtml(qid)} — ${escapeHtml(String(item.label || ''))}</strong><span class="muted">${escapeHtml(String(item.description || ''))}</span>`;
        button.addEventListener('click', () => {
          dom.domainQid.value = qid;
          state.domain_qid = qid;
          saveAutosave('wikidata_pick');
          renderAll(false);
          setMessage('success', `Выбран домен <code>${escapeHtml(qid)}</code>.`);
        });
        dom.wikidataResults.appendChild(button);
      });
    } catch (err) {
      dom.wikidataStatus.innerHTML = `<span style="color:#b91c1c">Ошибка поиска: ${escapeHtml(String(err))}</span>`;
    }
  }

  function escapeHtml(value) {
    return String(value || '').replace(/[&<>"']/g, (ch) => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;' }[ch]));
  }

  function escapeAttr(value) {
    return escapeHtml(value);
  }

  document.addEventListener('input', (event) => {
    const target = event.target;
    if (!(target instanceof HTMLElement)) return;
    if (target.matches('[data-paper-field]')) {
      const index = Number(target.getAttribute('data-paper-index'));
      const field = target.getAttribute('data-paper-field');
      if (!Number.isFinite(index) || !field) return;
      state.papers[index][field] = field === 'year' ? Number(target.value || 0) : target.value;
      saveAutosave(`paper:${field}`);
      return;
    }
    if (target.matches('[data-step-field]')) {
      const index = Number(target.getAttribute('data-step-index'));
      const field = target.getAttribute('data-step-field');
      if (!Number.isFinite(index) || !field) return;
      state.steps[index][field] = target.value;
      renderEdgesMatrix();
      saveAutosave(`step:${field}`);
      return;
    }
    if (target.matches('[data-source-field]')) {
      const stepIndex = Number(target.getAttribute('data-step-index'));
      const sourceIndex = Number(target.getAttribute('data-source-index'));
      const field = target.getAttribute('data-source-field');
      if (!Number.isFinite(stepIndex) || !Number.isFinite(sourceIndex) || !field) return;
      state.steps[stepIndex].sources[sourceIndex][field] = target.value;
      saveAutosave(`source:${field}`);
      return;
    }
    if (target.matches('[data-condition-key]')) {
      const stepIndex = Number(target.getAttribute('data-step-index'));
      const key = target.getAttribute('data-condition-key');
      if (!Number.isFinite(stepIndex) || !key) return;
      state.steps[stepIndex].conditions[key] = target.value;
      saveAutosave(`condition:${key}`);
      return;
    }
    if ([dom.expertLast, dom.expertFirst, dom.expertPat, dom.topicInput, dom.cutoffYearInput, dom.filenameInput, dom.domainQuery, dom.domainLanguage, dom.domainQid].includes(target)) {
      syncTopFieldsToState();
      state.steps = state.steps.map((step, idx) => normalizeStep(step, idx + 1, state.domain_qid));
      refreshMeta();
      if (target === dom.domainQid) {
        renderSteps();
      }
      saveAutosave(`field:${target.id}`);
    }
  });

  document.addEventListener('change', (event) => {
    const target = event.target;
    if (!(target instanceof HTMLElement)) return;
    if (target.matches('.edge-check')) {
      const from = Number(target.getAttribute('data-edge-from'));
      const to = Number(target.getAttribute('data-edge-to'));
      const key = `${from}->${to}`;
      const exists = state.edges.some((pair) => pair[0] === from && pair[1] === to);
      if (target.checked && !exists) {
        state.edges.push([from, to]);
      } else if (!target.checked && exists) {
        state.edges = state.edges.filter((pair) => !(pair[0] === from && pair[1] === to));
      }
      state.edges = normalizeEdges(state.edges, state.steps.length);
      renderEdgesMatrix();
      saveAutosave(`edge:${key}`);
      return;
    }
    if (target === dom.loadDraftInput && target.files && target.files[0]) {
      loadDraftFromFile(target.files[0]);
    }
  });

  document.addEventListener('click', (event) => {
    const target = event.target;
    if (!(target instanceof HTMLElement)) return;
    const removePaperIndex = target.getAttribute('data-remove-paper');
    if (removePaperIndex !== null) {
      removePaper(Number(removePaperIndex));
      return;
    }
    const removeStepIndex = target.getAttribute('data-remove-step');
    if (removeStepIndex !== null) {
      removeStep(Number(removeStepIndex));
      return;
    }
    const addSourceIndex = target.getAttribute('data-add-source');
    if (addSourceIndex !== null) {
      addSource(Number(addSourceIndex));
      return;
    }
    const removeSourceRef = target.getAttribute('data-remove-source');
    if (removeSourceRef !== null) {
      const [stepIndex, sourceIndex] = removeSourceRef.split(':').map((part) => Number(part));
      removeSource(stepIndex, sourceIndex);
    }
  });

  dom.addPaperBtn.addEventListener('click', addPaper);
  dom.addStepBtn.addEventListener('click', addStep);
  dom.exportYamlBtn.addEventListener('click', exportYaml);
  dom.saveDraftBtn.addEventListener('click', saveDraftJson);
  dom.restoreBtn.addEventListener('click', () => restoreAutosave(true));
  dom.clearDraftBtn.addEventListener('click', () => {
    localStorage.removeItem(STORAGE_KEY);
    setAutosaveStatus('Автосохранение удалено', 'info');
    setMessage('warning', 'Локальный черновик удалён из браузера. Текущие поля на странице не очищены.');
  });
  dom.domainSearchBtn.addEventListener('click', searchWikidata);

  renderAll(false);
  restoreAutosave(false);
  setAutosaveStatus('Форма готова. Автосохранение включено.', 'info');
  </script>
</body>
</html>
'''


def build_task1_offline_form(output_path: str | Path, initial_state: dict[str, Any] | None = None, domain_configs: list[dict[str, Any]] | None = None) -> Path:
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)

    state = initial_state or _default_state()
    configs = domain_configs or [
        {
            "domain_id": "science",
            "title": "General scientific literature",
            "wikidata_qid": "Q336",
            "required_conditions": [],
            "source_path": "embedded",
        }
    ]

    html_text = (
        _HTML_TEMPLATE
        .replace('__PAGE_TITLE__', 'Task 1 offline form')
        .replace('__INITIAL_STATE__', json.dumps(state, ensure_ascii=False))
        .replace('__DOMAIN_CONFIGS__', json.dumps(configs, ensure_ascii=False))
        .replace('__STORAGE_KEY__', 'task1-offline-form-v1')
    )
    output.write_text(html_text, encoding='utf-8')
    return output


Ок. Теперь запускайте ячейку: «ФОРМА».


In [ ]:

# @title
# ФОРМА (запустите ячейку и заполните поля)
domain_configs = _load_domain_configs()

# -----------------------------
# UI helpers + autosave
# -----------------------------
FORM_STATE_SCHEMA = "trajectory_form_state_v3"
FORM_STATE_VAR_NAME = "_TRAJECTORY_FORM_SESSION_STATE_V3"
FORM_AUTOSAVE_PATH = Path(".trajectory_form_autosave.json")

_BOUND_AUTOSAVE_WIDGETS = set()
_BOUND_EDGE_LABEL_WIDGETS = set()
_form_state_restore_in_progress = False
_edge_buttons = {}     # from_step_id -> {to_step_id: ToggleButton}
_edge_sync_in_progress = False


def _clean_text_preview(text, limit=60):
    text = re.sub(r"\s+", " ", str(text or "").strip())
    if len(text) > limit:
        text = text[: max(1, limit - 1)].rstrip() + "…"
    return text


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return int(default)


def _update_autosave_status(message="", kind="info"):
    if "autosave_status" not in globals():
        return
    colors = {
        "info": "#555",
        "success": "#0a7f2e",
        "warning": "#9a6700",
        "error": "#c00",
    }
    color = colors.get(kind, colors["info"])
    autosave_status.value = f"<small style='color:{color}'>{message}</small>" if message else ""


def _bind_autosave(widget, names="value"):
    if widget is None:
        return
    key = (id(widget), tuple(names) if isinstance(names, (list, tuple, set)) else names)
    if key in _BOUND_AUTOSAVE_WIDGETS:
        return

    def _handler(change):
        if _form_state_restore_in_progress:
            return
        _save_form_state(reason=f"change:{change.get('name', 'value')}")

    try:
        widget.observe(_handler, names=names)
        _BOUND_AUTOSAVE_WIDGETS.add(key)
    except Exception:
        pass


def _bind_claim_refresh(widget):
    if widget is None:
        return
    key = id(widget)
    if key in _BOUND_EDGE_LABEL_WIDGETS:
        return

    def _handler(change):
        _refresh_edge_labels()
        if _form_state_restore_in_progress:
            return
        _save_form_state(reason="claim_changed")

    try:
        widget.observe(_handler, names="value")
        _BOUND_EDGE_LABEL_WIDGETS.add(key)
    except Exception:
        pass


def _wd_label(item):
    if not item:
        return ""
    qid = item.get("qid") or ""
    label = item.get("label") or ""
    desc = item.get("description") or ""
    tail = (f" — {desc}" if desc else "")
    return f"{qid} | {label}{tail}"


def _current_required_keys(domain_qid: str):
    domain_qid = (domain_qid or "").strip()
    cfg = next((d for d in domain_configs if str(d.get("wikidata_qid") or "").strip() == domain_qid), None)
    return (cfg or {}).get("required_conditions", []) or []


def _edge_step_label(step_id: int, short_limit=38):
    if not isinstance(step_id, int) or step_id < 1 or step_id > len(step_widgets):
        return str(step_id)
    claim = _clean_text_preview(step_widgets[step_id - 1]["claim"].value, limit=short_limit)
    return f"{step_id}: {claim}" if claim else str(step_id)



def _save_form_state(reason="autosave"):
    return None

# -----------------------------
# Основное
# -----------------------------
expert_last = W.Text(description="Фамилия*", placeholder="Иванов", layout=W.Layout(width="100%"))
expert_first = W.Text(description="Имя*", placeholder="Иван", layout=W.Layout(width="100%"))
expert_pat = W.Text(description="Отчество*", placeholder="Иванович (или '-')", value="-", layout=W.Layout(width="100%"))

topic = W.Text(description="Topic*", placeholder="Коротко: о чём траектория", layout=W.Layout(width="100%"))
cutoff_year = W.BoundedIntText(description="Cutoff year*", value=2020, min=1800, max=2100, layout=W.Layout(width="250px"))

for _w in [expert_last, expert_first, expert_pat, topic, cutoff_year]:
    _bind_autosave(_w)

# -----------------------------
# Домен (ТОЛЬКО Wikidata QID)
# -----------------------------
wd_query = W.Text(
    description="Wikidata search",
    placeholder="Введите домен (например: Science / Наука / Lithium-ion battery)",
    layout=W.Layout(width="100%"),
)
wd_language = W.Dropdown(
    options=[("Русский (ru)", "ru"), ("English (en)", "en")],
    value="ru",
    description="Lang",
    layout=W.Layout(width="250px"),
)
wd_search_btn = W.Button(description="🔎 Найти в Wikidata", button_style="info")
wd_results = W.Dropdown(options=[("Сначала нажмите «Найти в Wikidata»", "")], description="Выбор", layout=W.Layout(width="100%"))
wd_status = W.HTML(value="")
domain_qid = W.Text(description="Domain (QID)*", placeholder="Q336", layout=W.Layout(width="250px"))
domain_qid.disabled = True

for _w in [wd_query, wd_language]:
    _bind_autosave(_w)


def _set_wd_results_placeholder(qid="", status_html=None):
    qid = (qid or "").strip()
    if qid:
        wd_results.options = [(f"Текущий выбор: {qid}", qid)]
        try:
            wd_results.value = qid
        except Exception:
            pass
        if status_html is not None:
            wd_status.value = status_html
    else:
        wd_results.options = [("Сначала нажмите «Найти в Wikidata»", "")]
        try:
            wd_results.value = ""
        except Exception:
            pass
        if status_html is not None:
            wd_status.value = status_html


def _run_wd_search(_):
    q = wd_query.value.strip()
    if not q:
        wd_status.value = "<span style='color:#c00'>Введите запрос для поиска.</span>"
        return
    wd_status.value = "Ищу..."
    try:
        res = wikidata_search(q, language=wd_language.value, limit=15)
        if not res:
            wd_results.options = [("Ничего не найдено", "")]
            wd_status.value = "<span style='color:#c00'>Ничего не найдено. Уточните запрос/язык.</span>"
            _save_form_state(reason="wikidata_search_empty")
            return
        wd_results.options = [(_wd_label(x), x["qid"]) for x in res if x.get("qid")]
        wd_results.value = wd_results.options[0][1]
        wd_status.value = f"Найдено: {len(wd_results.options)}"
        _save_form_state(reason="wikidata_search")
    except Exception as e:
        wd_results.options = [("Ошибка поиска", "")]
        wd_status.value = f"<span style='color:#c00'>Ошибка: {e}</span>"


def _on_wd_pick(change):
    qid = (change["new"] or "").strip()
    domain_qid.value = qid
    if qid:
        wd_status.value = f"Выбрано: <b>{qid}</b>"
    req = _current_required_keys(qid)
    for sw in step_widgets:
        _ensure_required_condition_fields(sw, req)
    _rebuild_edges_ui()
    _save_form_state(reason="wikidata_pick")


wd_search_btn.on_click(_run_wd_search)
wd_results.observe(_on_wd_pick, names="value")

# -----------------------------
# Кнопка на Google Form
# -----------------------------
FORM_URL = "https://forms.gle/aWszGzcKcnai2UXv9"
form_btn = W.HTML(value=f"""<a href="{FORM_URL}" target="_blank" rel="noopener">
<button style="padding:8px 12px;border-radius:8px;border:1px solid #ddd;cursor:pointer;">
Открыть Google‑форму
</button></a>""")

autosave_status = W.HTML(value="<small style='color:#555'>Форма автосохраняется. GitHub token намеренно не сохраняется автоматически.</small>")

# -----------------------------
# Публикации (papers)
# -----------------------------
paper_widgets = []
papers_box = W.VBox([], layout=W.Layout(width="100%"))

add_paper_btn = W.Button(description="➕ Добавить paper", button_style="info")
clear_papers_btn = W.Button(description="🧹 Очистить papers", button_style="warning", layout=W.Layout(width="170px"))


def _make_paper_widget():
    pid = W.Text(description="id*", placeholder="doi:... / arxiv:... / openalex:... / url", layout=W.Layout(width="35%"))
    year = W.IntText(description="year*", value=0, layout=W.Layout(width="15%"))
    title = W.Text(description="title*", placeholder="Название статьи", layout=W.Layout(width="50%"))
    rm = W.Button(description="Удалить", button_style="warning", layout=W.Layout(width="110px"))

    for _w in [pid, year, title]:
        _bind_autosave(_w)

    row = W.HBox([pid, year, title, rm], layout=W.Layout(width="100%"))
    w = {"id": pid, "year": year, "title": title, "ui": row}

    def _remove(_=None):
        if w in paper_widgets:
            paper_widgets.remove(w)
            papers_box.children = [p["ui"] for p in paper_widgets]
        if not paper_widgets:
            _add_paper()
        _save_form_state(reason="remove_paper")

    rm.on_click(_remove)
    return w


def _add_paper(_=None, preset=None):
    w = _make_paper_widget()
    paper_widgets.append(w)
    papers_box.children = [p["ui"] for p in paper_widgets]
    if isinstance(preset, dict):
        w["id"].value = str(preset.get("id") or "")
        y = preset.get("year")
        try:
            w["year"].value = int(y) if y is not None else 0
        except Exception:
            w["year"].value = 0
        w["title"].value = str(preset.get("title") or "")
    _save_form_state(reason="add_paper")
    return w


def _clear_papers(_=None, keep_one=True):
    paper_widgets.clear()
    papers_box.children = []
    if keep_one:
        _add_paper()
    _save_form_state(reason="clear_papers")


add_paper_btn.on_click(_add_paper)
clear_papers_btn.on_click(_clear_papers)


def _collect_papers():
    out = []
    for idx, p in enumerate(paper_widgets, start=1):
        pid = (p["id"].value or "").strip()
        title = (p["title"].value or "").strip()
        year_val = int(p["year"].value or 0)

        # Пустая строка → пропускаем
        if not pid and not title and year_val == 0:
            continue

        if not pid or not title or year_val <= 0:
            raise ValueError(f"papers[{idx}]: заполните id, year и title (или удалите строку).")

        out.append({"id": pid, "year": year_val, "title": title})
    return out


papers_ui = W.VBox([
    W.HBox([add_paper_btn, clear_papers_btn]),
    papers_box,
], layout=W.Layout(width="100%"))

# -----------------------------
# Шаги + источники
# -----------------------------
step_widgets = []


def _fill_source_widget(src_widget, data):
    src = data if isinstance(data, dict) else {}
    t = str(src.get("type") or "text").strip().lower()
    if t == "figure":
        t = "image"
    if t not in ("text", "image", "table"):
        t = "text"
    src_widget["type"].value = t
    src_widget["source"].value = str(src.get("source") or "")
    src_widget["page"].value = str(src.get("page") or "")
    src_widget["locator"].value = str(src.get("locator") or src.get("figure_or_table") or "")
    src_widget["snippet_or_summary"].value = str(src.get("snippet_or_summary") or "")


def _make_source_widget(on_remove):
    src_type = W.Dropdown(options=[("Text", "text"), ("Image/Figure", "image"), ("Table", "table")], value="text", description="type", layout=W.Layout(width="250px"))
    src_id = W.Text(description="source*", placeholder="doi:... / arxiv:... / openalex:... / url", layout=W.Layout(width="100%"))
    src_page = W.Text(description="page", placeholder="например, 1 (опционально)", value="", layout=W.Layout(width="250px"))
    src_loc = W.Text(description="locator", placeholder="Figure 3 / Table 2 (для image/table)", layout=W.Layout(width="250px"))
    src_snip = W.Textarea(description="snippet*", placeholder="Цитата/выжимка/описание (минимум 1–2 предложения)", layout=W.Layout(width="100%", height="80px"))
    rm = W.Button(description="Удалить источник", button_style="warning", layout=W.Layout(width="180px"))

    for _w in [src_type, src_id, src_page, src_loc, src_snip]:
        _bind_autosave(_w)

    def _remove_clicked(_=None):
        on_remove()

    rm.on_click(_remove_clicked)
    box = W.VBox([
        W.HBox([src_type, src_page, src_loc]),
        src_id,
        src_snip,
        rm,
        W.HTML("<hr style='border:none;border-top:1px solid #eee'>"),
    ], layout=W.Layout(width="100%", border="1px solid #f0f0f0", padding="8px", margin="4px 0"))
    return {
        "ui": box,
        "type": src_type,
        "source": src_id,
        "page": src_page,
        "locator": src_loc,
        "snippet_or_summary": src_snip,
    }


def _ensure_required_condition_fields(step_widget, required_keys):
    cond_widgets = step_widget["conditions_widgets"]
    children = list(step_widget["conditions_box"].children)
    changed = False
    for k in required_keys or []:
        if k not in cond_widgets:
            w = W.Text(description=f"{k}*", placeholder="unknown", value="unknown", layout=W.Layout(width="100%"))
            _bind_autosave(w)
            cond_widgets[k] = w
            children.append(w)
            changed = True
    if changed:
        step_widget["conditions_box"].children = children


def _set_step_conditions(step_widget, cond):
    cond = cond if isinstance(cond, dict) else {}
    base_defaults = {
        "system": "unknown",
        "environment": "unknown",
        "protocol": "unknown",
        "notes": "",
    }
    for key, default in base_defaults.items():
        if key in step_widget["conditions_widgets"]:
            step_widget["conditions_widgets"][key].value = str(cond.get(key) or default)

    for k in cond.keys():
        if k not in step_widget["conditions_widgets"]:
            w = W.Text(description=f"{k}*", placeholder="unknown", value="", layout=W.Layout(width="100%"))
            _bind_autosave(w)
            step_widget["conditions_widgets"][k] = w
            step_widget["conditions_box"].children = list(step_widget["conditions_box"].children) + [w]

    for k, w in step_widget["conditions_widgets"].items():
        if k not in base_defaults:
            w.value = str(cond.get(k) or "")


def _set_step_sources(step_widget, src_list):
    step_widget["sources"].clear()
    step_widget["sources_box"].children = []
    src_list = src_list if isinstance(src_list, list) and src_list else [{}]
    for src_data in src_list:
        src_widget = step_widget["add_source"]()
        _fill_source_widget(src_widget, src_data if isinstance(src_data, dict) else {})


def _make_step_widget(step_id: int):
    claim = W.Textarea(description="claim*", placeholder="Утверждение шага", layout=W.Layout(width="100%", height="70px"))

    # sources section
    sources_box = W.VBox([], layout=W.Layout(width="100%"))
    add_source_btn = W.Button(description="➕ Добавить источник", button_style="info")

    _bind_autosave(claim)
    _bind_claim_refresh(claim)

    sources = []

    def _add_source(_=None, preset=None):
        def _remove_this():
            if src in sources:
                sources.remove(src)
            sources_box.children = [s["ui"] for s in sources]
            _save_form_state(reason="remove_source")

        src = _make_source_widget(on_remove=_remove_this)
        sources.append(src)
        sources_box.children = [s["ui"] for s in sources]
        if isinstance(preset, dict):
            _fill_source_widget(src, preset)
        _save_form_state(reason="add_source")
        return src

    add_source_btn.on_click(_add_source)
    # start with 1 source
    _add_source()

    # conditions
    system = W.Text(description="system*", placeholder="unknown", value="unknown", layout=W.Layout(width="100%"))
    environment = W.Text(description="environment*", placeholder="unknown", value="unknown", layout=W.Layout(width="100%"))
    protocol = W.Text(description="protocol*", placeholder="unknown", value="unknown", layout=W.Layout(width="100%"))
    notes = W.Text(description="notes", placeholder="", value="", layout=W.Layout(width="100%"))

    for _w in [system, environment, protocol, notes]:
        _bind_autosave(_w)

    conditions_box = W.VBox([
        W.HTML("<b>Conditions</b> (если в статье не указано — пишите <code>unknown</code>)"),
        system, environment, protocol, notes,
    ], layout=W.Layout(width="100%"))

    inference = W.Textarea(description="inference*", placeholder="Логический вывод", layout=W.Layout(width="100%", height="70px"))
    next_q = W.Textarea(description="next_question*", placeholder="Вопрос для следующего шага", layout=W.Layout(width="100%", height="70px"))

    for _w in [inference, next_q]:
        _bind_autosave(_w)

    section = W.VBox([
        claim,
        W.HTML("<b>Sources</b> (можно добавить несколько)"),
        add_source_btn,
        sources_box,
        conditions_box,
        inference,
        next_q,
    ], layout=W.Layout(width="100%"))

    sw = {
        "step_id": step_id,
        "ui": section,
        "claim": claim,
        "sources": sources,
        "add_source": _add_source,
        "sources_box": sources_box,
        "conditions_widgets": {
            "system": system,
            "environment": environment,
            "protocol": protocol,
            "notes": notes,
        },
        "conditions_box": conditions_box,
        "inference": inference,
        "next_question": next_q,
    }
    # Add required keys if domain already selected
    req = _current_required_keys(domain_qid.value)
    _ensure_required_condition_fields(sw, req)
    return sw


steps_accordion = W.Accordion(children=[], layout=W.Layout(width="100%"))
add_step_btn = W.Button(description="➕ Добавить шаг", button_style="info")
del_step_btn = W.Button(description="🗑 Удалить последний шаг", button_style="warning")
_bind_autosave(steps_accordion, names="selected_index")


def _rebuild_steps(keep_n=1):
    global step_widgets
    keep_n = max(1, int(keep_n or 1))
    step_widgets = []
    children = []
    for i in range(1, keep_n + 1):
        sw = _make_step_widget(i)
        step_widgets.append(sw)
        children.append(sw["ui"])
    steps_accordion.children = children
    for i in range(len(children)):
        steps_accordion.set_title(i, f"Шаг {i+1}")
    steps_accordion.selected_index = 0 if children else None
    _rebuild_edges_ui()
    _save_form_state(reason="rebuild_steps")


def _add_step(_=None):
    n = len(step_widgets)
    sw = _make_step_widget(n + 1)
    step_widgets.append(sw)
    steps_accordion.children = list(steps_accordion.children) + [sw["ui"]]
    steps_accordion.set_title(len(steps_accordion.children) - 1, f"Шаг {n+1}")
    steps_accordion.selected_index = len(steps_accordion.children) - 1
    _rebuild_edges_ui()
    _save_form_state(reason="add_step")


def _del_step(_=None):
    n = len(step_widgets)
    if n <= 1:
        return
    step_widgets.pop()
    steps_accordion.children = steps_accordion.children[:-1]
    if steps_accordion.children:
        steps_accordion.selected_index = min(n - 2, len(steps_accordion.children) - 1)
    _rebuild_edges_ui()
    _save_form_state(reason="delete_step")


add_step_btn.on_click(_add_step)
del_step_btn.on_click(_del_step)

# -----------------------------
# Связи между шагами (edges)
# UX: один клик = выбрать связь, повторный клик = снять связь
# -----------------------------
edges_state = {}       # from_step_id -> list[to_step_id]
edges_box = W.VBox([], layout=W.Layout(width="100%"))
edges_preview = W.Textarea(description="Edges preview", layout=W.Layout(width="100%", height="90px"))
edges_preview.disabled = True

clear_all_edges_btn = W.Button(description="🧹 Очистить все связи", button_style="warning", layout=W.Layout(width="190px"))


def _edges_from_state():
    edges = []
    for a, bs in sorted(edges_state.items(), key=lambda x: x[0]):
        seen = set()
        for b in bs:
            if a == b:
                continue
            if b in seen:
                continue
            seen.add(b)
            edges.append([a, b])
    return edges


def _update_edges_preview():
    edges_preview.value = str([tuple(x) for x in _edges_from_state()])


def _sync_edge_button_style(btn):
    btn.button_style = "info" if btn.value else ""
    btn.icon = "check" if btn.value else ""


def _sync_step_edges_from_buttons(step_id):
    buttons = _edge_buttons.get(step_id, {})
    selected = [to_step for to_step, btn in sorted(buttons.items()) if btn.value]
    edges_state[step_id] = selected


def _set_edges(edges_list):
    """Применить список ребёр (ориентированный граф) к UI."""
    global _edge_sync_in_progress
    n = len(step_widgets)

    for i in range(1, n + 1):
        edges_state[i] = []

    for pair in (edges_list or []):
        try:
            a = int(pair[0])
            b = int(pair[1])
        except Exception:
            continue
        if 1 <= a <= n and 1 <= b <= n and a != b:
            if b not in edges_state[a]:
                edges_state[a].append(b)

    _edge_sync_in_progress = True
    try:
        for step_id, buttons in _edge_buttons.items():
            chosen = set(edges_state.get(step_id, []))
            for to_step, btn in buttons.items():
                desired = to_step in chosen
                if btn.value != desired:
                    btn.value = desired
                _sync_edge_button_style(btn)
    finally:
        _edge_sync_in_progress = False

    for step_id in list(_edge_buttons.keys()):
        _sync_step_edges_from_buttons(step_id)

    _update_edges_preview()


def _clear_all_edges(_=None):
    _set_edges([])
    _save_form_state(reason="clear_all_edges")


clear_all_edges_btn.on_click(_clear_all_edges)


def _refresh_edge_labels():
    for _from_step, mapping in _edge_buttons.items():
        for to_step, btn in mapping.items():
            btn.description = _edge_step_label(to_step, short_limit=34)
            btn.tooltip = _edge_step_label(to_step, short_limit=120)


def _rebuild_edges_ui():
    global _edge_buttons
    existing_edges = _edges_from_state()
    _edge_buttons = {}
    edges_state.clear()

    n = len(step_widgets)
    children = [
        W.HTML("<b>Связи между шагами</b> (опционально). Это <u>ориентированный</u> граф: нажмите на шаг один раз, чтобы выбрать связь; нажмите повторно, чтобы снять её."),
        W.HBox([clear_all_edges_btn]),
    ]

    for i in range(1, n + 1):
        opts = [j for j in range(1, n + 1) if j != i]

        if not opts:
            children.append(W.HTML(f"<b>{i} →</b> нет других шагов для связи"))
            edges_state[i] = []
            continue

        buttons = []
        _edge_buttons[i] = {}

        for to_step in opts:
            btn = W.ToggleButton(
                value=False,
                description=_edge_step_label(to_step, short_limit=34),
                tooltip=_edge_step_label(to_step, short_limit=120),
                layout=W.Layout(width="auto", min_width="150px", margin="0 6px 6px 0"),
            )
            _sync_edge_button_style(btn)

            def _handler(change, step_id=i, _btn=btn):
                if change.get("name") != "value":
                    return
                _sync_edge_button_style(_btn)
                if _edge_sync_in_progress:
                    return
                _sync_step_edges_from_buttons(step_id)
                _update_edges_preview()
                _save_form_state(reason="toggle_edge")

            btn.observe(_handler, names="value")
            _edge_buttons[i][to_step] = btn
            buttons.append(btn)

        clear_btn = W.Button(description="Снять всё для шага", layout=W.Layout(width="170px"))
        def _clear_step(_=None, step_id=i):
            global _edge_sync_in_progress
            buttons_map = _edge_buttons.get(step_id, {})
            _edge_sync_in_progress = True
            try:
                for btn in buttons_map.values():
                    btn.value = False
                    _sync_edge_button_style(btn)
            finally:
                _edge_sync_in_progress = False
            _sync_step_edges_from_buttons(step_id)
            _update_edges_preview()
            _save_form_state(reason="clear_step_edges")

        clear_btn.on_click(_clear_step)

        buttons_wrap = W.Box(
            buttons,
            layout=W.Layout(display="flex", flex_flow="row wrap", width="100%"),
        )

        children.append(
            W.VBox([
                W.HTML(f"<b>{i} →</b> выберите целевые шаги"),
                buttons_wrap,
                W.HBox([clear_btn]),
            ], layout=W.Layout(width="100%", border="1px solid #f0f0f0", padding="8px", margin="4px 0"))
        )
        edges_state[i] = []

    edges_box.children = children
    _set_edges(existing_edges)
    _refresh_edge_labels()
    _update_edges_preview()


# Инициализация: начинаем минимум с 1 шага и 1 paper
_rebuild_steps(keep_n=1)
if not paper_widgets:
    _add_paper()

# -----------------------------
# Autosave state helpers
# -----------------------------
def _collect_raw_papers():
    return [
        {
            "id": str(p["id"].value or ""),
            "year": _safe_int(p["year"].value, default=0),
            "title": str(p["title"].value or ""),
        }
        for p in paper_widgets
    ]


def _collect_raw_steps():
    steps_out = []
    for idx, sw in enumerate(step_widgets, start=1):
        srcs = []
        for s in sw["sources"]:
            srcs.append({
                "type": s["type"].value,
                "source": str(s["source"].value or ""),
                "page": str(s["page"].value or ""),
                "locator": str(s["locator"].value or ""),
                "snippet_or_summary": str(s["snippet_or_summary"].value or ""),
            })
        cond = {k: str(w.value or "") for k, w in sw["conditions_widgets"].items()}
        steps_out.append({
            "step_id": idx,
            "claim": str(sw["claim"].value or ""),
            "sources": srcs,
            "conditions": cond,
            "inference": str(sw["inference"].value or ""),
            "next_question": str(sw["next_question"].value or ""),
        })
    return steps_out


def _build_form_state():
    return {
        "_schema": FORM_STATE_SCHEMA,
        "_saved_at": _now_utc_z(),
        "expert": {
            "last_name": str(expert_last.value or ""),
            "first_name": str(expert_first.value or ""),
            "patronymic": str(expert_pat.value or ""),
        },
        "topic": str(topic.value or ""),
        "cutoff_year": _safe_int(cutoff_year.value, default=2020),
        "domain_query": str(wd_query.value or ""),
        "domain_language": str(wd_language.value or "ru"),
        "domain_qid": str(domain_qid.value or ""),
        "papers": _collect_raw_papers(),
        "steps": _collect_raw_steps(),
        "edges": _edges_from_state(),
        "selected_step_index": steps_accordion.selected_index if steps_accordion.selected_index is not None else 0,
        "filename": str(filename.value or "") if "filename" in globals() else "",
        "github": {
            "repo": str(gh_repo.value or "") if "gh_repo" in globals() else "",
            "branch": str(gh_branch.value or "") if "gh_branch" in globals() else "",
            "path": str(gh_path.value or "") if "gh_path" in globals() else "",
            "message": str(gh_commit_msg.value or "") if "gh_commit_msg" in globals() else "",
            # token намеренно не сохраняем
        },
    }


def _save_form_state(reason="autosave"):
    if _form_state_restore_in_progress:
        return
    try:
        state = _build_form_state()
        state["_saved_reason"] = str(reason or "autosave")
        globals()[FORM_STATE_VAR_NAME] = state
        try:
            FORM_AUTOSAVE_PATH.write_text(
                json.dumps(state, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            _update_autosave_status(
                f"Форма автосохранена ({state.get('_saved_at', '')}). GitHub token не сохраняется автоматически.",
                kind="info",
            )
        except Exception as file_exc:
            _update_autosave_status(
                f"Форма сохранена в памяти сессии, но не удалось записать файл автосохранения: {file_exc}",
                kind="warning",
            )
    except Exception as e:
        _update_autosave_status(f"Не удалось обновить автосохранение: {e}", kind="warning")


def _load_saved_form_state():
    state = globals().get(FORM_STATE_VAR_NAME)
    if isinstance(state, dict) and state.get("_schema") == FORM_STATE_SCHEMA:
        restored = dict(state)
        restored["_loaded_from"] = "памяти текущей сессии"
        return restored

    if FORM_AUTOSAVE_PATH.exists():
        try:
            state = json.loads(FORM_AUTOSAVE_PATH.read_text(encoding="utf-8"))
            if isinstance(state, dict):
                state["_loaded_from"] = f"файла {FORM_AUTOSAVE_PATH.name}"
                return state
        except Exception:
            pass
    return None


def _normalize_loaded_sources(step: dict):
    if isinstance(step.get("sources"), list):
        return step.get("sources") or []
    ev = step.get("evidence")
    if isinstance(ev, dict) and ev:
        t = str(ev.get("type") or "text").strip().lower()
        if t == "figure":
            t = "image"
        return [{
            "type": t,
            "source": ev.get("source", ""),
            "page": ev.get("page", ""),
            "locator": ev.get("figure_or_table", "") or "",
            "snippet_or_summary": ev.get("snippet_or_summary", ""),
        }]
    return []


def _doc_to_form_state(doc: dict):
    doc = doc if isinstance(doc, dict) else {}
    expert = doc.get("expert") or {}
    return {
        "_schema": FORM_STATE_SCHEMA,
        "expert": {
            "last_name": str(expert.get("last_name") or ""),
            "first_name": str(expert.get("first_name") or ""),
            "patronymic": str(expert.get("patronymic") or ""),
        },
        "topic": str(doc.get("topic") or ""),
        "cutoff_year": _safe_int(doc.get("cutoff_year"), default=2020),
        "domain_query": str(wd_query.value or ""),
        "domain_language": str(wd_language.value or "ru"),
        "domain_qid": str(doc.get("domain") or "").strip(),
        "papers": [p for p in (doc.get("papers") or []) if isinstance(p, dict)] or [{}],
        "steps": [s for s in (doc.get("steps") or []) if isinstance(s, dict)] or [{}],
        "edges": doc.get("edges") or [],
        "selected_step_index": 0,
        "filename": str(filename.value or "") if "filename" in globals() else "",
        "github": {
            "repo": str(gh_repo.value or "") if "gh_repo" in globals() else "",
            "branch": str(gh_branch.value or "") if "gh_branch" in globals() else "",
            "path": str(gh_path.value or "") if "gh_path" in globals() else "",
            "message": str(gh_commit_msg.value or "") if "gh_commit_msg" in globals() else "",
        },
    }


def _apply_form_state(state: dict, restored_from="автосохранения"):
    global _form_state_restore_in_progress
    if not isinstance(state, dict):
        return False

    expert = state.get("expert") or {}
    github_state = state.get("github") or {}
    step_rows = state.get("steps") or [{}]
    if not isinstance(step_rows, list) or len(step_rows) == 0:
        step_rows = [{}]

    paper_rows = state.get("papers") or [{}]
    if not isinstance(paper_rows, list) or len(paper_rows) == 0:
        paper_rows = [{}]

    _form_state_restore_in_progress = True
    try:
        expert_last.value = str(expert.get("last_name") or "")
        expert_first.value = str(expert.get("first_name") or "")
        expert_pat.value = str(expert.get("patronymic") or "-")

        topic.value = str(state.get("topic") or "")
        cutoff_year.value = _safe_int(state.get("cutoff_year"), default=2020)

        wd_query.value = str(state.get("domain_query") or "")
        lang = str(state.get("domain_language") or "ru")
        wd_language.value = lang if lang in {"ru", "en"} else "ru"
        domain_qid.value = str(state.get("domain_qid") or "").strip()
        if domain_qid.value:
            _set_wd_results_placeholder(domain_qid.value)
        else:
            _set_wd_results_placeholder("")

        _clear_papers(keep_one=False)
        for row in paper_rows:
            _add_paper(preset=row if isinstance(row, dict) else {})

        _rebuild_steps(keep_n=len(step_rows))

        for idx, row in enumerate(step_rows, start=1):
            row = row if isinstance(row, dict) else {}
            sw = step_widgets[idx - 1]
            sw["claim"].value = str(row.get("claim") or "")
            sw["inference"].value = str(row.get("inference") or "")
            sw["next_question"].value = str(row.get("next_question") or "")
            _set_step_conditions(sw, row.get("conditions") or {})
            _set_step_sources(sw, _normalize_loaded_sources(row))

        _rebuild_edges_ui()
        _set_edges(state.get("edges") or [])

        if "filename" in globals():
            filename.value = str(state.get("filename") or "")
        if "gh_repo" in globals():
            gh_repo.value = str(github_state.get("repo") or gh_repo.value or "")
            gh_branch.value = str(github_state.get("branch") or gh_branch.value or "main")
            gh_path.value = str(github_state.get("path") or "")
            gh_commit_msg.value = str(github_state.get("message") or "")

        sel_idx = state.get("selected_step_index")
        if isinstance(sel_idx, int) and 0 <= sel_idx < len(step_widgets):
            steps_accordion.selected_index = sel_idx
        elif len(step_widgets):
            steps_accordion.selected_index = 0

        if domain_qid.value:
            wd_status.value = (
                f"<span style='color:#0a7f2e'>Форма восстановлена из {restored_from}. "
                "При необходимости можно заново выполнить поиск в Wikidata.</span>"
            )
        else:
            wd_status.value = f"<span style='color:#0a7f2e'>Форма восстановлена из {restored_from}.</span>"
    finally:
        _form_state_restore_in_progress = False

    _refresh_edge_labels()
    _save_form_state(reason="restore_complete")
    return True


def _restore_saved_form_state():
    state = _load_saved_form_state()
    if not state:
        _update_autosave_status(
            "Форма автосохраняется. GitHub token намеренно не сохраняется автоматически.",
            kind="info",
        )
        return False
    source = state.get("_loaded_from") or "автосохранения"
    ok = _apply_form_state(state, restored_from=source)
    if ok:
        _update_autosave_status(
            f"Форма восстановлена из {source}. GitHub token не автосохраняется.",
            kind="success",
        )
    return ok

# -----------------------------
# Загрузка YAML (продолжить редактирование формы)
# -----------------------------
yaml_upload = W.FileUpload(accept=".yaml,.yml", multiple=False)
yaml_load_btn = W.Button(description="📥 Загрузить YAML в форму", button_style="info")
yaml_load_out = W.Output()


def _first_uploaded_item(upload_value):
    if not upload_value:
        return None
    if isinstance(upload_value, dict):
        items = list(upload_value.values())
    elif isinstance(upload_value, (list, tuple)):
        items = list(upload_value)
    else:
        items = [upload_value]
    return items[0] if items else None


def _uploaded_item_content_bytes(uploaded):
    if uploaded is None:
        return None
    content = uploaded.get("content") if isinstance(uploaded, dict) else getattr(uploaded, "content", None)
    if content is None:
        return None
    if hasattr(content, "tobytes"):
        content = content.tobytes()
    elif isinstance(content, memoryview):
        content = content.tobytes()
    elif isinstance(content, bytearray):
        content = bytes(content)
    return content if isinstance(content, bytes) else bytes(content)


def _apply_doc_to_form(doc: dict):
    state = _doc_to_form_state(doc)
    return _apply_form_state(state, restored_from="YAML файла")


def _load_yaml_into_form(_=None):
    yaml_load_out.clear_output()
    with yaml_load_out:
        try:
            if not yaml_upload.value:
                print("❌ Выберите YAML файл (.yaml/.yml).")
                return
            uploaded = _first_uploaded_item(yaml_upload.value)
            content = _uploaded_item_content_bytes(uploaded)
            if content is None:
                print("❌ Не удалось прочитать содержимое файла.")
                return
            text = content.decode("utf-8", errors="replace")
            doc = yaml.safe_load(text)
            if not isinstance(doc, dict):
                print("❌ YAML должен содержать словарь верхнего уровня.")
                return
            _apply_doc_to_form(doc)
            _save_form_state(reason="load_yaml")
            print("✅ YAML загружен. Можно продолжать редактирование.")
        except Exception as e:
            print("❌ Ошибка чтения YAML:", e)


yaml_load_btn.on_click(_load_yaml_into_form)

yaml_load_ui = W.VBox([
    W.HTML("<b>Продолжить редактирование</b> — загрузите сохранённый .yaml и форма заполнится автоматически:"),
    W.HBox([yaml_upload, yaml_load_btn]),
    yaml_load_out,
], layout=W.Layout(width="100%"))

# -----------------------------
# Экспорт
# -----------------------------
filename = W.Text(description="filename", placeholder="Оставьте пустым — сгенерируем автоматически", layout=W.Layout(width="100%"))
generate_btn = W.Button(description="✅ Проверить и собрать YAML", button_style="success")
offline_form_btn = W.Button(description="⬇️ Скачать автономную форму", button_style="primary")
out = W.Output()
offline_form_out = W.Output()

_bind_autosave(filename)


def _collect_doc():
    # Build steps
    steps_out = []
    for idx, sw in enumerate(step_widgets, start=1):
        # sources
        srcs = []
        for s in sw["sources"]:
            d = {
                "type": s["type"].value,
                "source": s["source"].value.strip(),
                "snippet_or_summary": s["snippet_or_summary"].value.strip(),
            }
            p = (s["page"].value or "").strip()
            if p:
                d["page"] = p
            loc = (s["locator"].value or "").strip()
            if loc:
                d["locator"] = loc
            srcs.append(d)
        # conditions
        cond = {}
        for k, w in sw["conditions_widgets"].items():
            cond[k] = (w.value or "").strip()
        steps_out.append({
            "step_id": idx,
            "claim": (sw["claim"].value or "").strip(),
            "sources": srcs,
            "conditions": cond,
            "inference": (sw["inference"].value or "").strip(),
            "next_question": (sw["next_question"].value or "").strip(),
        })

    doc = {
        "artifact_version": 2,
        "topic": (topic.value or "").strip(),
        "domain": (domain_qid.value or "").strip(),
        "cutoff_year": int(cutoff_year.value or 0),
        "papers": _collect_papers(),
        "steps": steps_out,
        "edges": _edges_from_state(),
        "generated_at": _now_utc_z(),
    }
    doc = finalize_doc_with_ids(doc, expert_last.value, expert_first.value, expert_pat.value)
    return doc


def _export(_=None):
    _save_form_state(reason="before_export")
    out.clear_output()
    with out:
        try:
            doc = _collect_doc()
            required = _current_required_keys(doc.get("domain"))
            errs = _validate_trajectory(doc, required)
            if errs:
                print("❌ Есть ошибки. Введённые значения сохранены; исправьте поля и попробуйте снова.")
                for line in _format_errors_for_humans(errs):
                    print("-", line)
                _update_autosave_status(
                    "Проверка обнаружила ошибки, но все текущие данные формы сохранены.",
                    kind="warning",
                )
                return

            yml = yaml.safe_dump(doc, sort_keys=False, allow_unicode=True)
            print("✅ Ок. YAML собран. Ниже — превью:")
            print("\n" + yml[:4000] + ("\n... (truncated)" if len(yml) > 4000 else ""))

            # filename
            fname = (filename.value or "").strip()
            if not fname:
                fname = f"{doc.get('submission_id','trajectory')}.yaml"
            if not fname.lower().endswith((".yaml", ".yml")):
                fname += ".yaml"
            path = Path(fname).resolve()
            path.write_text(yml, encoding="utf-8")

            print(f"\n📄 Файл сохранён: {path}")
            _update_autosave_status(
                f"YAML успешно собран и сохранён ({path.name}).",
                kind="success",
            )
            if IN_COLAB:
                files.download(str(path))

        except Exception as e:
            print("❌ Ошибка:", e)
            _update_autosave_status(f"Во время экспорта возникла ошибка: {e}", kind="error")


generate_btn.on_click(_export)


def _download_offline_form(_=None):
    _save_form_state(reason="before_offline_form_export")
    offline_form_out.clear_output()
    with offline_form_out:
        try:
            current_state = _build_form_state()
            topic_slug = _slugify(current_state.get("topic") or "task1_reasoning_trajectories", max_len=80)
            path = Path(f"{topic_slug}__offline_form.html").resolve()
            build_task1_offline_form(path, initial_state=current_state, domain_configs=domain_configs)
            print(f"📄 Автономная форма сохранена: {path}")
            _update_autosave_status(
                f"Автономная HTML-форма подготовлена ({path.name}).",
                kind="success",
            )
            if IN_COLAB:
                files.download(str(path))
        except Exception as e:
            print("❌ Ошибка:", e)
            _update_autosave_status(f"Во время подготовки автономной формы возникла ошибка: {e}", kind="error")


offline_form_btn.on_click(_download_offline_form)

# -----------------------------
# GitHub upload (опционально): создаёт коммит в указанной ветке
# -----------------------------
gh_repo = W.Text(description="repo", value="https://github.com/top-papers/top-papers-graph", placeholder="owner/repo или https://github.com/owner/repo", layout=W.Layout(width="100%"))
gh_branch = W.Text(description="branch", value="main", layout=W.Layout(width="250px"))
gh_path = W.Text(description="path", placeholder="data/experts/trajectories/<submission_id>.yaml", layout=W.Layout(width="100%"))
gh_token = W.Password(description="token", placeholder="GitHub token (Contents: RW)", layout=W.Layout(width="100%"))
gh_commit_msg = W.Text(description="message", placeholder="Commit message", layout=W.Layout(width="100%"))
gh_upload_btn = W.Button(description="⬆️ Загрузить YAML в GitHub", button_style="info")
gh_out = W.Output()

for _w in [gh_repo, gh_branch, gh_path, gh_commit_msg]:
    _bind_autosave(_w)
# gh_token специально не автосохраняем


def _gh_upload(_=None):
    _save_form_state(reason="before_github_upload")
    gh_out.clear_output()
    with gh_out:
        try:
            token = (gh_token.value or "").strip()
            if not token:
                print("❌ Укажите token (Fine-grained: Contents Read+Write).")
                return

            repo_str = (gh_repo.value or "").strip()

            # Поддерживаем ввод как "owner/repo", так и полного URL вида https://github.com/owner/repo
            m = re.search(r"github\.com/([^/\s]+)/([^/\s]+?)(?:\.git)?/?$", repo_str)
            if m:
                owner, repo = m.group(1).strip(), m.group(2).strip()
            else:
                if "/" not in repo_str:
                    print("❌ repo должен быть в формате owner/repo или https://github.com/owner/repo")
                    return
                owner, repo = repo_str.split("/", 1)
                owner, repo = owner.strip(), repo.strip()
                if repo.endswith(".git"):
                    repo = repo[:-4]
            branch = (gh_branch.value or "main").strip() or "main"

            doc = _collect_doc()
            required = _current_required_keys(doc.get("domain"))
            errs = _validate_trajectory(doc, required)
            if errs:
                print("❌ Есть ошибки — сначала исправьте их (кнопка «Проверить и собрать YAML» покажет детали).")
                for line in _format_errors_for_humans(errs):
                    print("-", line)
                return

            yml = yaml.safe_dump(doc, sort_keys=False, allow_unicode=True)
            path_in_repo = (gh_path.value or "").strip()
            path_in_repo = path_in_repo.lstrip("/")
            if not path_in_repo:
                path_in_repo = f"data/experts/trajectories/{doc.get('submission_id','trajectory')}.yaml"

            msg = (gh_commit_msg.value or "").strip()
            if not msg:
                msg = f"Add trajectory {doc.get('submission_id','trajectory')}"

            url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path_in_repo}"
            headers = {
                "Authorization": f"Bearer {token}",
                "Accept": "application/vnd.github+json",
                "X-GitHub-Api-Version": "2022-11-28",
            }

            # Узнаём sha, если файл уже существует (иначе создадим новый)
            sha = None
            r = requests.get(url, headers=headers, params={"ref": branch})
            if r.status_code == 200:
                sha = (r.json() or {}).get("sha")
            elif r.status_code != 404:
                print("❌ Не удалось проверить файл в GitHub:", r.status_code)
                print(r.text[:2000])
                return

            payload = {
                "message": msg,
                "content": base64.b64encode(yml.encode("utf-8")).decode("ascii"),
                "branch": branch,
            }
            if sha:
                payload["sha"] = sha

            r2 = requests.put(url, headers=headers, json=payload)
            if r2.status_code not in (200, 201):
                print("❌ Ошибка загрузки в GitHub:", r2.status_code)
                if r2.status_code == 404:
                    print("Это обычно означает одно из следующих:")
                    print("  • неверный repo (owner/repo) или URL")
                    print("  • у токена нет доступа к этому репозиторию (fine‑grained token должен быть разрешён для repo)")
                    print("  • у вас нет прав на запись в upstream — используйте fork и затем PR")
                print(r2.text[:2000])
                return

            data = r2.json() or {}
            file_url = ((data.get("content") or {}).get("html_url") or "").strip()
            commit_url = ((data.get("commit") or {}).get("html_url") or "").strip()

            print("✅ Готово: YAML закоммичен в GitHub.")
            if file_url:
                display(W.HTML(f"<a href='{file_url}' target='_blank' rel='noopener'>Открыть файл</a>"))
            if commit_url:
                display(W.HTML(f"<a href='{commit_url}' target='_blank' rel='noopener'>Открыть коммит</a>"))
            print("Если нужно — создайте PR из ветки вручную в интерфейсе GitHub.")
            _update_autosave_status("GitHub upload завершён успешно.", kind="success")

        except Exception as e:
            print("❌ Неожиданная ошибка:", e)
            _update_autosave_status(f"Ошибка GitHub upload: {e}", kind="error")


gh_upload_btn.on_click(_gh_upload)

# -----------------------------
# Layout
# -----------------------------
ui = W.VBox([
    W.HTML("<h3>Trajectory form (artifact v2)</h3>"),
    autosave_status,
    W.HBox([form_btn]),
    W.HTML("<hr>"),
    yaml_load_ui,
    W.HTML("<hr>"),
    W.HTML("<b>Эксперт</b>"),
    expert_last, expert_first, expert_pat,
    W.HTML("<b>Основное</b>"),
    topic, cutoff_year,
    W.HTML("<b>Домен (Wikidata)</b>"),
    wd_query,
    W.HBox([wd_language, wd_search_btn]),
    wd_results,
    wd_status,
    domain_qid,
    W.HTML("<b>Публикации</b>"),
    papers_ui,
    W.HTML("<b>Шаги</b>"),
    W.HBox([add_step_btn, del_step_btn]),
    steps_accordion,
    W.HTML("<b>Связи</b>"),
    edges_box,
    edges_preview,
    W.HTML("<b>Экспорт</b>"),
    filename,
    W.HBox([generate_btn, offline_form_btn]),
    out,
    offline_form_out,
    W.HTML("<b>GitHub (опционально)</b>"),
    gh_repo,
    W.HBox([gh_branch]),
    gh_path,
    gh_commit_msg,
    gh_token,
    gh_upload_btn,
    gh_out,
], layout=W.Layout(width="100%"))

_restore_saved_form_state()
display(ui)


# Как получить токен GitHub (если вы загружаете YAML прямо из блокнота)

Этот раздел нужен **только** для кнопки «Загрузить YAML в GitHub» в конце формы.

## Fine-grained token (рекомендуется)
1. GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens
2. Repository access: **Only selected repositories** → выберите нужный репозиторий
3. Permissions:
   - **Contents: Read and write**

Сохраните токен и **скопируйте его** (GitHub показывает его один раз).

## Куда вставить
Вставьте токен в поле **token** в блоке «GitHub (опционально)» в конце формы.



# Частые вопросы (FAQ)

## 1) Я вижу код и мне страшно
Это нормально. Вам не нужно его понимать. Делайте только:
- запуск ячеек
- заполнение формы
- нажатие кнопок

## 2) Обязательные условия (temperature, soc_window, chemistry…) — откуда они взялись?
Некоторые домены требуют конкретные граничные условия на каждом шаге.  
В этом блокноте они подхватываются из конфигов домена (если они найдены).  
Если в статье это не указано — пишите `unknown`.

## 3) Файл YAML «не проходит» проверку
Проверьте:
- есть ли **6–12 шагов**
- заполнены ли **claim / evidence / conditions / inference / next_question** на каждом шаге
- заполнены ли обязательные `conditions.<ключ>` (если домен их требует)

## 4) GitHub выдаёт ошибку
Самые частые причины:
- токен без прав на запись (нужен доступ **Contents: Read and write**)
- защищённая main‑ветка (в таком случае создайте отдельную ветку и PR)
- нет доступа к репозиторию

Если не получается — используйте:
- «⬇️ Скачать YAML» → отправить через централизованный канал
- «💾 Google Drive» → поделиться ссылкой
